# Stuff+ / Pitching+ Pipeline v4 — Diablos Rojos Hackathon

1. **FIX critico**: el "Season holdout" de v3 tenia `tr_mask` y `te_mask`
   ambos con `== last_season` -> entrenaba y evaluaba sobre la MISMA
   temporada (leakage total). Corregido a `tr_mask = year != last_season`.
2. **Validacion Primaria vs Secundaria**: pitcher/season/altitud son
   evidencia principal (holdouts reales, sin leakage). El split aleatorio
   pasa a ser un chequeo de sanidad secundario, explicitamente etiquetado
   como tal (pitches del mismo pitcher pueden caer en train Y test ahi).
3. **Location+ como capa propia**: se construye un ablation acumulativo
   Stuff -> +Location -> +Arsenal -> +Contexto -> +Altitud, mostrando el
   aporte independiente de cada capa (antes Location+ solo se probaba
   con/sin dentro de Pitching+ completo, nunca aislada).
4. **4 clasificadores**: `target_weak_contact`, `target_hard_contact`,
   `target_barrel`, `target_groundball` (usando ExitSpeed/Angle, solo sobre
   `is_batted==1`).
5. **Perfil ideal de pitcher Diablos**: SHAP sobre el modelo Pitching+,
   mas comparacion de features fisicas/arsenal entre el cuartil de mejores
   vs peores pitches predichos -> tabla para scouting.
6. **Robustez con GroupKFold**: ademas del holdout puntual de pitcher, se
   reporta media +/- desviacion estandar de RMSE/AUC en 4 folds agrupados
   por pitcher, tanto para el modelo de regresion como para los clasificadores.
7. **Optuna omitido**: se usan directamente los mejores hiperparametros ya
   encontrados (40 trials, RMSE CV = 0.259268), aplicados de forma uniforme
   a todos los modelos de regresion de este notebook.


## Guía de lectura

Este cuaderno reúne **todo el código del proyecto** (modelos, validaciones, análisis de altitud, perfil ideal y exportaciones del dashboard) en el orden en que se ejecutó. 
| Parte | Contenido | Informe técnico |
|---|---|---|
| **A** | Configuración, carga y limpieza, variable objetivo (xRunValue), arsenal y capas de variables | Secc. 2–4 |
| **B** | Validación: holdouts por lanzador, temporada y altitud; split aleatorio de sanidad; GroupKFold | Secc. 5 |
| **C** | Clasificadores auxiliares (whiff, chase, strike, out, contacto, barrel, rodado) | Secc. 5.3 |
| **D** | Efecto de la altitud: Δ de Stuff+ y Pitching+, test dentro de lanzador | Secc. 6–7 |
| **E** | Perfil ideal (SHAP y cuartiles), estabilidad entre temporadas, selección del score | Secc. 8 y 5 |
| **F** | Exportación de modelos y rejilla de predicciones del dashboard (versiones v1–v4) | Anexo |
| **G** | Resultados finales y tablas del informe: scores, Δ por lanzador, validaciones finales, riesgo, top 3 | Secc. 5–8 |

**Orden de ejecución.** Las celdas se ejecutan de arriba abajo; las Partes F y G usan variables creadas en A–E (`df_cleaned`, `X_full`, `res_full`, `BEST_PARAMS`, etc).

**Datos.** El archivo de entrada (`stuff_model_df.parquet`).

**Convención de nombres.** `Stuff_plus_pure` = Stuff+ (solo física); `Pitching_plus_raw` = Pitching+ sin ajuste por altitud; `Pitching_plus_adj` = con ajuste por altitud; sufijo `_cn` = neutral al conteo (y a los outs en Pitching+).

# PARTE A · Configuración, datos y variables
*Informe técnico: Secciones 2, 3 y 4.* Entorno, hiperparámetros (`BEST_PARAMS`), limpieza física, imputación de altitud, objetivo xRunValue, variables de arsenal y capas de X.

In [1]:
import sys
print(sys.executable)

C:\Users\jorge\baseball_clean\Scripts\python.exe


In [2]:
import numpy as np
import pandas as pd
import shap
from sklearn.metrics import (
    brier_score_loss, log_loss, mean_squared_error, r2_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, GroupShuffleSplit, GroupKFold
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier, XGBRegressor
from scipy.stats import spearmanr, pearsonr

pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

# Mejores hiperparametros ya encontrados con Optuna (40 trials, CV agrupada por pitcher)
BEST_PARAMS = {
    'n_estimators': 275, 'max_depth': 7, 'learning_rate': 0.017621094782145158,
    'subsample': 0.8453461218551805, 'colsample_bytree': 0.80483950787895,
    'min_child_weight': 6, 'reg_alpha': 8.480087597597214,
    'reg_lambda': 1.1239131100512512, 'gamma': 0.05764198174478805,
    'random_state': RANDOM_STATE, 'n_jobs': -1,
}

def evaluate_regression(X_tr, y_tr, X_te, y_te, label="", params=None):
    p = dict(params) if params is not None else dict(BEST_PARAMS)
    m = XGBRegressor(**p)
    m.fit(X_tr, y_tr)
    pred = m.predict(X_te)
    return {"label": label,
            "rmse": float(np.sqrt(mean_squared_error(y_te, pred))),
            "r2": float(r2_score(y_te, pred)),
            "model": m, "pred": pred}


## 1. Carga y limpieza fisica

In [3]:
df = pd.read_parquet("stuff_model_df.parquet")
print(f"Dataset: {df.shape}")

stuff_features_core = [
    "RelSpeed", "EffectiveVelo", "ZoneSpeed", "SpinRate", "SpinAxis",
    "RelHeight", "RelSide", "Extension", "VertBreak", "InducedVertBreak",
    "HorzBreak", "VertRelAngle", "HorzRelAngle", "VertApprAngle",
    "HorzApprAngle", "ZoneTime",
]
complete_mask = df[stuff_features_core].notna().all(axis=1)
df_cleaned = df.loc[complete_mask].copy()

valid_mask = (
    (df_cleaned["RelSpeed"] >= 50) & (df_cleaned["RelSpeed"] <= 105)
    & (df_cleaned["Extension"] >= 3.0) & (df_cleaned["Extension"] <= 8.5)
    & (df_cleaned["VertBreak"] >= -90.0) & (df_cleaned["VertBreak"] <= 30.0)
    & (df_cleaned["InducedVertBreak"] >= -35.0) & (df_cleaned["InducedVertBreak"] <= 35.0)
    & (df_cleaned["ZoneTime"] >= 0.35) & (df_cleaned["ZoneTime"] <= 0.65)
    & (df_cleaned["RelHeight"].between(2.5, 8.0))
    & (df_cleaned["RelSide"].between(-5.0, 5.0))
)
df_cleaned = df_cleaned.loc[valid_mask].copy()

lhp_mask = df_cleaned["PitcherThrows"] == "Left"
for col in ["RelSide", "HorzBreak", "HorzRelAngle", "HorzApprAngle"]:
    df_cleaned.loc[lhp_mask, col] = df_cleaned.loc[lhp_mask, col] * -1

df_cleaned["EffectiveVelo"] = pd.to_numeric(df_cleaned["EffectiveVelo"], errors="coerce")
print(f"Filas tras limpieza: {len(df_cleaned)} (de {len(df)})")


Dataset: (635002, 84)
Filas tras limpieza: 632769 (de 635002)


## 2. Imputacion de `altitude_category`

In [4]:
alt_col = "altitude_category"
game_to_alt = (
    df_cleaned.dropna(subset=[alt_col]).groupby("game_anon_id")[alt_col]
    .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
)
df_cleaned[alt_col] = df_cleaned[alt_col].fillna(df_cleaned["game_anon_id"].map(game_to_alt))

if df_cleaned[alt_col].isna().sum() > 0:
    py = (df_cleaned.dropna(subset=[alt_col]).groupby(["pitcher_anon_id", "year"])[alt_col]
          .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan))
    mask_n = df_cleaned[alt_col].isna()
    m = df_cleaned.loc[mask_n, ["pitcher_anon_id", "year"]].merge(
        py.rename("alt"), left_on=["pitcher_anon_id", "year"], right_index=True, how="left")
    df_cleaned.loc[mask_n, alt_col] = m["alt"].values

if df_cleaned[alt_col].isna().sum() > 0:
    df_cleaned[alt_col] = df_cleaned[alt_col].fillna(df_cleaned[alt_col].mode().iloc[0])

assert df_cleaned[alt_col].isna().sum() == 0
alt_order = {"No Altitude": 0, "Medium Altitude": 1, "Extreme Altitude": 2}
df_cleaned["altitude_ord"] = df_cleaned[alt_col].map(alt_order)
print(f"Distribucion final: {df_cleaned[alt_col].value_counts().to_dict()}")


Distribucion final: {'No Altitude': 296125, 'Extreme Altitude': 182424, 'Medium Altitude': 154220}


## 3. Target `xRunValue`

In [5]:
half_inning_key = ["game_anon_id", "Inning", "Top/Bottom"]

n_before = len(df_cleaned)
df_cleaned = df_cleaned[df_cleaned["Outs"].astype(int) < 3].copy()
print(f"Filas removidas con Outs>=3: {n_before - len(df_cleaned)}")

df = df_cleaned.sort_values(half_inning_key + ["PitchUID"]).reset_index(drop=True).copy()
for c in ["Outs", "Balls", "Strikes"]:
    df[c] = df[c].astype(int)

event_cols = ["single", "double", "triple", "home_run",
              "base_on_balls", "is_hit_by_pitch", "strikeout"]
other_out = (df["is_ball_in_play"] == 1) & (df["is_hit"] == 0)
df["is_pa_end"] = ((df[event_cols].sum(axis=1) > 0) | other_out).astype(int)

df["runs_after"] = (
    df.groupby(half_inning_key)["RunsScored"]
    .transform(lambda s: s.shift(-1).fillna(0)[::-1].cumsum()[::-1])
)

re_table = df.groupby(["Outs", "Balls", "Strikes"])["runs_after"].mean().rename("RE").reset_index()
for c in ["Outs", "Balls", "Strikes"]:
    re_table[c] = re_table[c].astype(int)
re_table = pd.concat(
    [re_table, pd.DataFrame({"Outs": [3], "Balls": [0], "Strikes": [0], "RE": [0.0]})], ignore_index=True)
re_table = re_table.drop_duplicates(subset=["Outs", "Balls", "Strikes"], keep="first").reset_index(drop=True)

if "OutsOnPlay" in df.columns:
    outs_gained = df["OutsOnPlay"].fillna(0).astype(int)
else:
    outs_gained = (df["strikeout"] == 1).astype(int) + other_out.astype(int)

df["Outs_next"] = np.where(df["is_pa_end"] == 1, (df["Outs"] + outs_gained).clip(upper=3), df["Outs"]).astype(int)
df["Balls_next"] = np.where(df["is_pa_end"] == 1, 0, df["Balls"]).astype(int)
df["Strikes_next"] = np.where(df["is_pa_end"] == 1, 0, df["Strikes"]).astype(int)

df = df.merge(re_table.rename(columns={"RE": "RE_current"}), on=["Outs", "Balls", "Strikes"], how="left")
df = df.merge(re_table.rename(columns={"Outs": "Outs_next", "Balls": "Balls_next",
                                        "Strikes": "Strikes_next", "RE": "RE_next"}),
              on=["Outs_next", "Balls_next", "Strikes_next"], how="left")

assert df["RE_current"].isna().sum() == 0
assert df["RE_next"].isna().sum() == 0

df["xRunValue"] = df["RE_next"] - df["RE_current"] + df["RunsScored"]
df_cleaned = df.copy()
print(f"xRunValue: mean={df_cleaned['xRunValue'].mean():+.4f}, std={df_cleaned['xRunValue'].std():.4f}")
print(f"df_cleaned: {df_cleaned.shape}")


Filas removidas con Outs>=3: 8


C:\Users\jorge\AppData\Local\Temp\ipykernel_18084\3306440833.py:17: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(half_inning_key)["RunsScored"]


xRunValue: mean=+0.0111, std=0.2651
df_cleaned: (632761, 93)


## 4. Arsenal (fastball primaria real + reset por PA )

In [6]:
df_cleaned = df_cleaned.sort_values(half_inning_key + ["PitchUID"]).reset_index(drop=True)
df_cleaned["pa_num"] = df_cleaned.groupby(half_inning_key)["is_pa_end"].cumsum().sub(
    df_cleaned["is_pa_end"]).astype(int)
pa_group = ["pitcher_anon_id", "game_anon_id", "Inning", "Top/Bottom", "pa_num"]

fb_types = ["Four-Seam", "Sinker", "Cutter", "FourSeam", "4-Seam", "Fastball"]
ap = df_cleaned["AutoPitchType"].astype(str)
fb_present = [t for t in fb_types if t in ap.unique()]

fb_pool = df_cleaned[df_cleaned["AutoPitchType"].isin(fb_present)]
primary_fb_type = (fb_pool.groupby("pitcher_anon_id")["AutoPitchType"]
                    .agg(lambda s: s.mode().iloc[0]).rename("primary_fb_type"))
fb_pool_tagged = fb_pool.merge(primary_fb_type, on="pitcher_anon_id", how="left")
fb_pool_primary = fb_pool_tagged[fb_pool_tagged["AutoPitchType"] == fb_pool_tagged["primary_fb_type"]]
fb_stats = (fb_pool_primary.groupby("pitcher_anon_id")
            .agg(fb_velo=("RelSpeed", "mean"), fb_ivb=("InducedVertBreak", "mean"), fb_hb=("HorzBreak", "mean"))
            .reset_index())

df_cleaned = df_cleaned.drop(columns=["fb_velo", "fb_ivb", "fb_hb", "primary_fb_type"], errors="ignore")
df_cleaned = df_cleaned.merge(fb_stats, on="pitcher_anon_id", how="left")
df_cleaned = df_cleaned.merge(primary_fb_type, on="pitcher_anon_id", how="left")

df_cleaned["velo_diff_vs_fb"] = df_cleaned["RelSpeed"] - df_cleaned["fb_velo"]
df_cleaned["ivb_diff_vs_fb"] = df_cleaned["InducedVertBreak"] - df_cleaned["fb_ivb"]
df_cleaned["hb_diff_vs_fb"] = df_cleaned["HorzBreak"] - df_cleaned["fb_hb"]

usage = (df_cleaned.groupby(["pitcher_anon_id", "AutoPitchType"]).size()
         .groupby(level=0).transform(lambda s: s / s.sum()).rename("pitch_usage_pct"))
if "pitch_usage_pct" in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=["pitch_usage_pct"])
df_cleaned = df_cleaned.join(usage, on=["pitcher_anon_id", "AutoPitchType"])
df_cleaned["pitch_usage_pct"] = df_cleaned["pitch_usage_pct"].fillna(0)

df_cleaned = df_cleaned.sort_values(
    ["pitcher_anon_id", "game_anon_id", "Inning", "Top/Bottom", "pa_num", "PitchUID"]).reset_index(drop=True)
release_feats = [c for c in ["RelSpeed", "RelSide", "RelHeight", "x0", "y0", "z0"] if c in df_cleaned.columns]
stds = df_cleaned[release_feats].std().replace(0, 1)
prev = df_cleaned.groupby(pa_group)[release_feats].shift(1)
diff_norm = (df_cleaned[release_feats] - prev) / stds
dist = np.sqrt((diff_norm ** 2).sum(axis=1))
df_cleaned["tunnel_similarity"] = 1 / (1 + dist)
df_cleaned["tunnel_similarity"] = df_cleaned["tunnel_similarity"].fillna(df_cleaned["tunnel_similarity"].median())

df_cleaned["prev_pitch_type"] = (
    df_cleaned.groupby(pa_group)["AutoPitchType"].shift(1).astype(object).fillna("None"))
df_cleaned = df_cleaned.drop(columns=["fb_velo", "fb_ivb", "fb_hb"], errors="ignore")
print("Arsenal listo.")


C:\Users\jorge\AppData\Local\Temp\ipykernel_18084\655910755.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_cleaned["pa_num"] = df_cleaned.groupby(half_inning_key)["is_pa_end"].cumsum().sub(
C:\Users\jorge\AppData\Local\Temp\ipykernel_18084\655910755.py:27: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  usage = (df_cleaned.groupby(["pitcher_anon_id", "AutoPitchType"]).size()
C:\Users\jorge\AppData\Local\Temp\ipykernel_18084\655910755.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior 

Arsenal listo.


## 5. Features y familias de X — Stuff -> +Location -> +Arsenal -> +Contexto -> +Altitud

Se construye la jerarquia COMPLETA en capas acumulativas. Esto le da a Location+ su propio punto de medicion
independiente.

In [7]:
categorical_cols = ["PitcherThrows", "BatterSide", "AutoPitchType", "prev_pitch_type"]
cat_present = [c for c in categorical_cols if c in df_cleaned.columns]

# 1) Los NaN de handedness no deben caer en la categoría base de drop_first
for c in ["PitcherThrows", "BatterSide"]:
    if c in df_cleaned.columns:
        df_cleaned[c] = df_cleaned[c].fillna("Unknown")

# 2) Features de platoon explícitas (los LHP tienen el horizontal espejado, PlateLocSide no)
if "PitcherThrows" in df_cleaned.columns and "BatterSide" in df_cleaned.columns:
    df_cleaned["platoon_same_hand"] = (df_cleaned["PitcherThrows"] == df_cleaned["BatterSide"]).astype(float)
    bat_sign = df_cleaned["BatterSide"].map({"Right": 1.0, "Left": -1.0}).fillna(0.0)
    # + = hacia la parte de adentro (inside) para el bateador
    df_cleaned["loc_inside_batter"] = -df_cleaned["PlateLocSide"] * bat_sign

# 3) dtype=float evita que las dummies se pierdan en select_dtypes
df_encoded = pd.get_dummies(df_cleaned, columns=cat_present, drop_first=True, dtype=float)
df_encoded["altitude_ord"] = df_cleaned["altitude_ord"].values
df_encoded["xRunValue"] = df_cleaned["xRunValue"].values

stuff_features = [
    "RelSpeed", "SpinRate", "SpinAxis", "InducedVertBreak", "HorzBreak",
    "Extension", "RelHeight", "RelSide",
    "EffectiveVelo", "ZoneSpeed", "SpeedDrop", "ZoneTime",
    "VertRelAngle", "HorzRelAngle", "VertApprAngle", "HorzApprAngle",
    "x0", "y0", "z0", "vx0", "vy0", "vz0", "ax0", "ay0", "az0",
    "pfxx", "pfxz",
    "PitchTrajectoryXc0", "PitchTrajectoryXc1", "PitchTrajectoryXc2",
    "PitchTrajectoryYc0", "PitchTrajectoryYc1", "PitchTrajectoryYc2",
    "PitchTrajectoryZc0", "PitchTrajectoryZc1", "PitchTrajectoryZc2",
]

location_features = ["PlateLocHeight", "PlateLocSide", "in_strike_zone"]
count_features = ["Balls", "Strikes"]
arsenal_features = ["velo_diff_vs_fb", "ivb_diff_vs_fb", "hb_diff_vs_fb", "pitch_usage_pct", "tunnel_similarity"]
extra_context_features = ["Outs", "Inning"]

cat_dummies = [c for c in df_encoded.columns if any(c.startswith(f"{x}_") for x in cat_present)]

def build_X(features, include_cat_dummies=True):
    cols = [c for c in features if c in df_encoded.columns]
    if include_cat_dummies:
        cols = cols + cat_dummies
    X = df_encoded[cols].select_dtypes(include=[np.number]).copy()
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    return X

# --- Capas acumulativas ---
X_stuff        = build_X(stuff_features, include_cat_dummies=False)  
X_loc_only     = build_X(location_features + count_features)          
X_stuff_loc    = build_X(stuff_features + location_features + count_features)
X_stuff_loc_ars = build_X(stuff_features + location_features + count_features + arsenal_features)
X_full_no_alt  = build_X(stuff_features + location_features + count_features + arsenal_features + extra_context_features)
X_full         = build_X(stuff_features + location_features + count_features + arsenal_features + extra_context_features + ["altitude_ord"])

print(f"X_stuff (fisico puro, sin dummies):  {X_stuff.shape}")
print(f"X_loc_only (Location+ aislado):       {X_loc_only.shape}")
print(f"X_stuff_loc:                          {X_stuff_loc.shape}")
print(f"X_stuff_loc_ars:                      {X_stuff_loc_ars.shape}")
print(f"X_full_no_alt (= Pitching+ sin alt):  {X_full_no_alt.shape}")
print(f"X_full (= Pitching+ completo):        {X_full.shape}")

X_stuff (fisico puro, sin dummies):  (632761, 36)
X_loc_only (Location+ aislado):       (632761, 35)
X_stuff_loc:                          (632761, 71)
X_stuff_loc_ars:                      (632761, 76)
X_full_no_alt (= Pitching+ sin alt):  (632761, 78)
X_full (= Pitching+ completo):        (632761, 79)


## 6. Split principal (para el chequeo de sanidad secundario — ver Seccion 8)

In [8]:
n_null = df_encoded["pitcher_anon_id"].isna().sum()
if n_null > 0:
    mask_valid = df_encoded["pitcher_anon_id"].notna().values
    df_encoded = df_encoded[mask_valid].reset_index(drop=True)
    df_cleaned = df_cleaned[mask_valid].reset_index(drop=True)
    for name in ["X_stuff", "X_loc_only", "X_stuff_loc", "X_stuff_loc_ars", "X_full_no_alt", "X_full"]:
        globals()[name] = globals()[name][mask_valid].reset_index(drop=True)

assert df_encoded["pitcher_anon_id"].isna().sum() == 0
print(f"Pitchers unicos: {df_encoded['pitcher_anon_id'].nunique()} | Filas: {len(df_encoded)}")

idx_train, idx_test = train_test_split(
    df_encoded.index, test_size=0.15, random_state=RANDOM_STATE, shuffle=True,
    stratify=df_encoded["altitude_ord"],
)
df_train, df_test = df_encoded.loc[idx_train], df_encoded.loc[idx_test]
y_train, y_test = df_train["xRunValue"], df_test["xRunValue"]

def split_all(X):
    return X.loc[idx_train], X.loc[idx_test]

X_train_stuff, X_test_stuff = split_all(X_stuff)
X_train_loc, X_test_loc = split_all(X_loc_only)
X_train_sl, X_test_sl = split_all(X_stuff_loc)
X_train_sla, X_test_sla = split_all(X_stuff_loc_ars)
X_train_full_noalt, X_test_full_noalt = split_all(X_full_no_alt)
X_train_full, X_test_full = split_all(X_full)

print(f"Train: {len(idx_train)} | Test: {len(idx_test)}")


Pitchers unicos: 1134 | Filas: 629756
Train: 535292 | Test: 94464


# PARTE B · Validación
-Holdouts por lanzador, temporada y altitud (secc. 7). -split aleatorio (secc. 8) chequeo de sanidad, porque los pitcheos de un mismo lanzador pueden caer en entrenamiento y prueba.

## 7. VALIDACION PRIMARIA — Pitcher / Season (CORREGIDO) / Altitud

Los tres modelos usan grupos que el modelo nunca vio en train (pitcher completo, temporada completa, o regimen
de altitud completo).

In [9]:
def evaluate_holdout(X_tr, y_tr, X_te, y_te, label):
    m = XGBRegressor(**BEST_PARAMS)
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    return {"label": label, "rmse": float(np.sqrt(mean_squared_error(y_te, p))), "r2": float(r2_score(y_te, p))}

def baseline_holdout(df_tr, df_te):
    tab = df_tr.groupby(["Outs", "Balls", "Strikes"])["xRunValue"].mean().rename("b").reset_index()
    m = df_te.merge(tab, on=["Outs", "Balls", "Strikes"], how="left")
    m["b"] = m["b"].fillna(df_tr["xRunValue"].mean())
    return float(np.sqrt(mean_squared_error(df_te["xRunValue"], m["b"])))

y_all = df_encoded["xRunValue"].values

# --- A. Pitcher holdout ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(X_full, groups=df_encoded["pitcher_anon_id"]))
res_ph = evaluate_holdout(X_full.iloc[tr_idx], y_all[tr_idx], X_full.iloc[te_idx], y_all[te_idx], "A. Pitcher holdout")
base_ph = baseline_holdout(df_encoded.iloc[tr_idx], df_encoded.iloc[te_idx])

# --- B. Season holdout (FIX: antes tr_mask y te_mask eran identicos) ---
seasons = sorted(df_cleaned["year"].unique())
last_season = seasons[-1]
tr_mask_season = (df_cleaned["year"] != last_season).values   # <- FIX: era "== last_season"
te_mask_season = (df_cleaned["year"] == last_season).values
print(f"Season holdout: train={tr_mask_season.sum()} filas (temporadas {[s for s in seasons if s != last_season]}), "
      f"test={te_mask_season.sum()} filas (temporada {last_season})")
res_sh = evaluate_holdout(X_full[tr_mask_season], y_all[tr_mask_season],
                           X_full[te_mask_season], y_all[te_mask_season], f"B. Season holdout ({last_season})")
base_sh = baseline_holdout(df_encoded[tr_mask_season], df_encoded[te_mask_season])

# --- C. Altitud/estadio holdout (proxy: Extreme Altitude) ---
alt = df_encoded["altitude_ord"].values
tr_mask_alt, te_mask_alt = alt != 2, alt == 2
res_ah = evaluate_holdout(X_full[tr_mask_alt], y_all[tr_mask_alt],
                           X_full[te_mask_alt], y_all[te_mask_alt], "C. Altitud/estadio holdout (proxy: Extreme)")
base_ah = baseline_holdout(df_encoded[tr_mask_alt], df_encoded[te_mask_alt])

print(f"\n{'Holdout PRIMARIO':40s} {'Modelo':>10s} {'Baseline':>10s} {'Delta':>10s} {'Gana?':>8s}")
print("-" * 82)
for r, b in [(res_ph, base_ph), (res_sh, base_sh), (res_ah, base_ah)]:
    delta = b - r["rmse"]
    print(f"{r['label']:40s} {r['rmse']:>10.5f} {b:>10.5f} {delta:>+10.5f} {'SI' if delta>0 else 'NO':>8s}")


Season holdout: train=497927 filas (temporadas [np.int32(2024), np.int32(2025)]), test=131829 filas (temporada 2026)

Holdout PRIMARIO                             Modelo   Baseline      Delta    Gana?
----------------------------------------------------------------------------------
A. Pitcher holdout                          0.26315    0.26547   +0.00233       SI
B. Season holdout (2026)                    0.25902    0.26112   +0.00210       SI
C. Altitud/estadio holdout (proxy: Extreme)    0.28167    0.28401   +0.00234       SI


## 8. VALIDACION SECUNDARIA — split aleatorio (chequeo de sanidad, NO evidencia principal)

Pitches del mismo pitcher pueden caer en train y test aqui. Se reporta solo
como confirmacion de que el modelo no tiene un bug obvio, no como prueba de
generalizacion — esa prueba la dan los 3 holdouts primarios de arriba.

In [10]:
res_stuff = evaluate_regression(X_train_stuff, y_train, X_test_stuff, y_test, "Stuff+ (fisico puro)")
res_loc = evaluate_regression(X_train_loc, y_train, X_test_loc, y_test, "Location+ (AISLADO)")
res_sl = evaluate_regression(X_train_sl, y_train, X_test_sl, y_test, "Stuff + Location+")
res_sla = evaluate_regression(X_train_sla, y_train, X_test_sla, y_test, "Stuff + Location+ + Arsenal")
res_full_noalt = evaluate_regression(X_train_full_noalt, y_train, X_test_full_noalt, y_test, "Pitching+ SIN altitud")
res_full = evaluate_regression(X_train_full, y_train, X_test_full, y_test, "Pitching+ COMPLETO (con altitud)")

base_tab = df_train.groupby(["Outs", "Balls", "Strikes"])["xRunValue"].mean().rename("b").reset_index()
base_te = df_test.merge(base_tab, on=["Outs", "Balls", "Strikes"], how="left")
base_te["b"] = base_te["b"].fillna(y_train.mean())
rmse_base = np.sqrt(mean_squared_error(y_test, base_te["b"]))

print(f"{'Modelo (capas acumulativas)':38s} {'RMSE':>10s} {'R2':>10s}")
print("-" * 60)
print(f"{'Baseline por conteo':38s} {rmse_base:>10.5f} {'':>10s}")
for r in [res_stuff, res_loc, res_sl, res_sla, res_full_noalt, res_full]:
    print(f"{r['label']:38s} {r['rmse']:>10.5f} {r['r2']:>10.5f}")

print(f"\nAporte independiente de Location+ (solo, vs baseline por conteo): "
      f"Delta RMSE = {rmse_base - res_loc['rmse']:+.5f}")
print(f"Aporte incremental de Location+ (dentro de Stuff+Location vs Stuff solo): "
      f"Delta RMSE = {res_stuff['rmse'] - res_sl['rmse']:+.5f}")
print(f"Aporte incremental de Arsenal (Stuff+Loc+Arsenal vs Stuff+Loc): "
      f"Delta RMSE = {res_sl['rmse'] - res_sla['rmse']:+.5f}")
print(f"Aporte incremental de Altitud (completo vs sin altitud): "
      f"Delta RMSE = {res_full_noalt['rmse'] - res_full['rmse']:+.5f}")


Modelo (capas acumulativas)                  RMSE         R2
------------------------------------------------------------
Baseline por conteo                       0.26463           
Stuff+ (fisico puro)                      0.26730    0.00397
Location+ (AISLADO)                       0.26652    0.00972
Stuff + Location+                         0.26642    0.01046
Stuff + Location+ + Arsenal               0.26641    0.01059
Pitching+ SIN altitud                     0.26274    0.03762
Pitching+ COMPLETO (con altitud)          0.26273    0.03772

Aporte independiente de Location+ (solo, vs baseline por conteo): Delta RMSE = -0.00190
Aporte incremental de Location+ (dentro de Stuff+Location vs Stuff solo): Delta RMSE = +0.00087
Aporte incremental de Arsenal (Stuff+Loc+Arsenal vs Stuff+Loc): Delta RMSE = +0.00002
Aporte incremental de Altitud (completo vs sin altitud): Delta RMSE = +0.00001


## 9. Robustez — GroupKFold (media +/- std por pitcher)

Un solo holdout de pitcher (Seccion 7A) puede ser optimista o pesimista por
azar en quienes caen en el fold de test. Aqui se repite 4 veces con distintos
grupos de pitchers y se reporta la variabilidad.

In [11]:
gkf = GroupKFold(n_splits=4)
groups_all = df_encoded["pitcher_anon_id"].values

rmses_full, r2s_full = [], []
for tr_idx, te_idx in gkf.split(X_full, y_all, groups=groups_all):
    m = XGBRegressor(**BEST_PARAMS)
    m.fit(X_full.iloc[tr_idx], y_all[tr_idx])
    p = m.predict(X_full.iloc[te_idx])
    rmses_full.append(np.sqrt(mean_squared_error(y_all[te_idx], p)))
    r2s_full.append(r2_score(y_all[te_idx], p))

print(f"Pitching+ completo, GroupKFold(4) por pitcher:")
print(f"  RMSE: {np.mean(rmses_full):.5f} +/- {np.std(rmses_full):.5f}  (folds: {[round(x,5) for x in rmses_full]})")
print(f"  R2:   {np.mean(r2s_full):.5f} +/- {np.std(r2s_full):.5f}  (folds: {[round(x,5) for x in r2s_full]})")


Pitching+ completo, GroupKFold(4) por pitcher:
  RMSE: 0.25975 +/- 0.00251  (folds: [np.float64(0.26051), np.float64(0.26042), np.float64(0.25563), np.float64(0.26245)])
  R2:   0.03991 +/- 0.00115  (folds: [0.03893, 0.03904, 0.04179, 0.03987])


# PARTE C · Clasificadores auxiliares
8 clasificadores (whiff, chase, called strike, out en la aparición AB, weak contact, contacto fuerte, barrel y groundout) con validación por lanzador.

## 10. Clasificadores multi-target COMPLETOS (los 6 de clasificacion que pide el hackathon)

`target_weak_contact`, `target_hard_contact`, `target_barrel`,
`target_groundball`, entrenados SOLO sobre `is_batted==1` (donde ExitSpeed/
Angle existen). Barrel usa una aproximacion de la formula de Statcast
(interpolacion EV/angulo); weak/hard contact usan umbrales de EV
razonables pero ajustables a altura—

In [12]:
print("Diagnostico de hit_type (para calibrar groundball):")
print(df_cleaned["hit_type"].value_counts(dropna=False).head(15))
print(f"\nFilas con ExitSpeed/Angle no nulos (is_batted): {df_cleaned['is_batted'].sum()}")


Diagnostico de hit_type (para calibrar groundball):
hit_type
None          447174
GroundBall     63354
FlyBall        49980
LineDrive      37502
Popup          29472
Bunt            2274
Name: count, dtype: int64

Filas con ExitSpeed/Angle no nulos (is_batted): 223957.0


In [13]:
# ============================================================
# 12A. DIAGNÓSTICO DE GROUNDBALL
# ============================================================

batted_mask = df_cleaned["is_batted"] == 1

print("Total de pitches:", len(df_cleaned))
print("Total de batted balls:", batted_mask.sum())

print("\nDistribución de hit_type:")
print(
    df_cleaned.loc[batted_mask, "hit_type"]
    .value_counts(dropna=False)
)

print("\nBatted balls sin hit_type:")
n_missing_hit_type = (
    batted_mask &
    df_cleaned["hit_type"].isna()
).sum()

print(n_missing_hit_type)

print("\nGroundBalls explícitos:")
n_groundball = (
    batted_mask &
    df_cleaned["hit_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("groundball")
).sum()

print(n_groundball)

print(
    f"\n% de batted balls con GroundBall explícito: "
    f"{100 * n_groundball / batted_mask.sum():.2f}%"
)

Total de pitches: 629756
Total de batted balls: 223957

Distribución de hit_type:
hit_type
GroundBall    63354
FlyBall       49980
None          41375
LineDrive     37502
Popup         29472
Bunt           2274
Name: count, dtype: int64

Batted balls sin hit_type:
41375

GroundBalls explícitos:
63354

% de batted balls con GroundBall explícito: 28.29%


In [14]:
# Diagnóstico específico para definir target_groundball

batted = df_cleaned["is_batted"] == 1

print("Total de lanzamientos bateados:", batted.sum())

print("\nDistribución de hit_type entre bolas bateadas:")
print(
    df_cleaned.loc[batted, "hit_type"]
    .value_counts(dropna=False)
)

print("\nBolas bateadas sin hit_type:")
print(
    df_cleaned.loc[
        batted & df_cleaned["hit_type"].isna()
    ].shape[0]
)

print("\nGroundballs explícitos:")
print(
    (
        batted &
        df_cleaned["hit_type"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("groundball")
    ).sum()
)

Total de lanzamientos bateados: 223957

Distribución de hit_type entre bolas bateadas:
hit_type
GroundBall    63354
FlyBall       49980
None          41375
LineDrive     37502
Popup         29472
Bunt           2274
Name: count, dtype: int64

Bolas bateadas sin hit_type:
41375

Groundballs explícitos:
63354


In [15]:
print(df_train["hit_type"].dropna().unique())

['GroundBall' 'LineDrive' 'FlyBall' 'Popup' 'Bunt']


In [16]:
def make_targets(df_):
    df_ = df_.copy()
    df_["target_whiff"] = (df_["is_whiff"] == 1).astype(int)
    df_["target_chase"] = (df_["swung_outside_strike_zone"] == 1).astype(int)
    df_["target_strike"] = (df_["is_called_strike"] == 1).astype(int)         # P(called strike)
    df_["target_out_pa"] = (
        (df_["is_pa_end"] == 1) &
        ((df_["strikeout"] == 1) | ((df_["is_ball_in_play"] == 1) & (df_["is_hit"] == 0)))
    ).astype(int)

    # --- Nuevos: solo tienen sentido donde hubo contacto (is_batted == 1) ---
    batted = df_["is_batted"] == 1

    # Groundball: clasificacion explicita desde hit_type (confirmado confiable: 63,354 filas)
    df_["target_groundball"] = (
        batted &
        df_["hit_type"].astype(str).str.strip().str.lower().eq("groundball")
    ).astype(int)

    # Hard / weak contact: umbrales de ExitSpeed (ajustables)
    df_["target_hard_contact"] = (batted & (df_["ExitSpeed"] >= 95)).astype(int)   # estandar MLB "hard-hit"
    df_["target_weak_contact"] = (batted & (df_["ExitSpeed"] < 80)).astype(int)    # umbral aproximado, calibrar

    # Barrel: aproximacion de la formula Statcast (EV>=98, rango de angulo que se ensancha con EV)
    def is_barrel(ev, la):
        if pd.isna(ev) or pd.isna(la) or ev < 98:
            return False
        if ev >= 116:
            return 8 <= la <= 50
        frac = (ev - 98) / (116 - 98)
        low = 26 - frac * (26 - 8)
        high = 30 + frac * (50 - 30)
        return low <= la <= high

    df_["target_barrel"] = 0
    df_.loc[batted, "target_barrel"] = df_.loc[batted].apply(
        lambda row: int(is_barrel(row["ExitSpeed"], row["Angle"])), axis=1
    )
    return df_
df_train = make_targets(df_train)
df_test = make_targets(df_test)

print("target_groundball" in df_train.columns)  # debe salir True ahora

True


In [17]:
print("target_groundball" in df_train.columns)
print([c for c in df_train.columns if c.startswith("target_")])

True
['target_whiff', 'target_chase', 'target_strike', 'target_out_pa', 'target_groundball', 'target_hard_contact', 'target_weak_contact', 'target_barrel']


In [18]:
print("\nDistribución de target_groundball:")
print(df_train["target_groundball"].value_counts())

print("\nProporción de groundballs entre bolas bateadas:")
print(
    df_train.loc[df_train["is_batted"] == 1, "target_groundball"]
    .mean()
)


Distribución de target_groundball:
target_groundball
0    481384
1     53908
Name: count, dtype: int64

Proporción de groundballs entre bolas bateadas:
0.28316743269862116


In [19]:
all_targets = ["target_whiff", "target_chase", "target_strike", "target_out_pa",
               "target_weak_contact", "target_hard_contact", "target_barrel", "target_groundball"]
contact_only_targets = {"target_weak_contact", "target_hard_contact", "target_barrel", "target_groundball"}

clf_models = {}
calibration_results = {}

print(f"{'Target':22s} {'AUC':>8s} {'LogLoss':>10s} {'Brier':>8s} {'Base%':>7s} {'n_train':>9s}")
print("-" * 70)
for t in all_targets:
    if t in contact_only_targets:
        train_mask = df_train["is_batted"] == 1
        test_mask = df_test["is_batted"] == 1
    else:
        train_mask = pd.Series(True, index=df_train.index)
        test_mask = pd.Series(True, index=df_test.index)

    y_tr_t = df_train.loc[train_mask, t]
    y_te_t = df_test.loc[test_mask, t]
    X_tr_t = X_full.loc[df_train.loc[train_mask].index]
    X_te_t = X_full.loc[df_test.loc[test_mask].index]

    clf = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
    clf.fit(X_tr_t, y_tr_t)
    p = clf.predict_proba(X_te_t)[:, 1]
    clf_models[t] = (clf, train_mask, test_mask)

    auc, ll, brier = roc_auc_score(y_te_t, p), log_loss(y_te_t, p), brier_score_loss(y_te_t, p)
    print(f"{t:22s} {auc:>8.4f} {ll:>10.4f} {brier:>8.4f} {y_te_t.mean()*100:>6.1f}% {train_mask.sum():>9d}")
    calibration_results[t] = calibration_curve(y_te_t, p, n_bins=10, strategy="quantile")

Target                      AUC    LogLoss    Brier   Base%   n_train
----------------------------------------------------------------------
target_whiff             0.7588     0.3093   0.0912   11.3%    535292
target_chase             0.9170     0.2948   0.0964   22.3%    535292
target_strike            0.8819     0.2878   0.0917   15.4%    535292
target_out_pa            0.8067     0.3583   0.1136   16.4%    535292
target_weak_contact      0.6314     0.6494   0.2289   40.5%    190375
target_hard_contact      0.6681     0.3726   0.1115   13.4%    190375
target_barrel            0.7027     0.0776   0.0156    1.6%    190375
target_groundball        0.6412     0.5686   0.1923   28.1%    190375


### 10.1 Robustez de clasificadores — GroupKFold por pitcher (media +/- std de AUC)

In [20]:
print(f"{'Target':22s} {'AUC medio':>12s} {'+/- std':>10s}")
print("-" * 48)
for t in all_targets:
    if t in contact_only_targets:
        sub_mask = df_encoded.index.isin(df_cleaned[df_cleaned["is_batted"] == 1].index)
    else:
        sub_mask = np.ones(len(df_encoded), dtype=bool)

    df_sub = df_encoded[sub_mask].copy()
    df_sub_targets = make_targets(df_cleaned[sub_mask].reset_index(drop=True))
    y_sub = df_sub_targets[t].values
    X_sub = X_full[sub_mask].reset_index(drop=True)
    groups_sub = df_sub_targets["pitcher_anon_id"].values

    if pd.Series(y_sub).nunique() < 2:
        continue

    aucs = []
    for tr_idx, te_idx in gkf.split(X_sub, y_sub, groups=groups_sub):
        if pd.Series(y_sub[tr_idx]).nunique() < 2 or pd.Series(y_sub[te_idx]).nunique() < 2:
            continue
        clf = XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             random_state=RANDOM_STATE, n_jobs=-1, eval_metric="logloss")
        clf.fit(X_sub.iloc[tr_idx], y_sub[tr_idx])
        p = clf.predict_proba(X_sub.iloc[te_idx])[:, 1]
        aucs.append(roc_auc_score(y_sub[te_idx], p))

    if aucs:
        print(f"{t:22s} {np.mean(aucs):>12.4f} {np.std(aucs):>10.4f}")


Target                    AUC medio    +/- std
------------------------------------------------
target_whiff                 0.7526     0.0018
target_chase                 0.9154     0.0006
target_strike                0.8812     0.0010
target_out_pa                0.8058     0.0014
target_weak_contact          0.6302     0.0027
target_hard_contact          0.6644     0.0023
target_barrel                0.6805     0.0068
target_groundball            0.6372     0.0020


La correlación negativa entre tasa de groundballs y score Pitching+ se reduce a aproximadamente la mitad en Extreme Altitude (ρ=−0.11) respecto al conjunto general (ρ=−0.22), sugiriendo que la capacidad de inducir groundballs gana valor relativo en condiciones de altitud, consistente con H3. No obstante, la comparación es descriptiva (muestras distintas, no una prueba formal de diferencia entre correlaciones) y el signo se mantiene negativo en ambos casos — los pitchers dominantes por velocidad/whiff siguen siendo, en promedio, los de mejor Pitching+ incluso en altitud.

# PARTE D · Efecto de la altitud
Δ de Stuff+ y Pitching+ entre altitud extrema y sin altitud, y prueba dentro de lanzador con el score final ajustado.

## 11. Delta Stuff+/Pitching+ por altitud (Stuff+ fisico puro vs Pitching+ completo)

In [21]:
df_eval = df_cleaned.copy()

df_eval["stuff_pred_pure"] = res_stuff["model"].predict(X_stuff)
mu_s, sd_s = df_eval["stuff_pred_pure"].mean(), df_eval["stuff_pred_pure"].std()
df_eval["Stuff_plus_pure"] = 100 - 10 * (df_eval["stuff_pred_pure"] - mu_s) / sd_s

df_eval["pitching_pred_raw"] = res_full_noalt["model"].predict(X_full_no_alt)
mu_p, sd_p = df_eval["pitching_pred_raw"].mean(), df_eval["pitching_pred_raw"].std()
df_eval["Pitching_plus_raw"] = 100 - 10 * (df_eval["pitching_pred_raw"] - mu_p) / sd_p

for score_col, label in [("Stuff_plus_pure", "STUFF+ FISICO PURO"), ("Pitching_plus_raw", "PITCHING+ COMPLETO")]:
    print(f"\nDelta {label} por tipo de pitch (Extreme vs No Altitude):")
    print("=" * 70)
    for pt in df_eval["AutoPitchType"].unique():
        sub = df_eval[df_eval["AutoPitchType"] == pt]
        no_alt = sub.loc[sub["altitude_category"] == "No Altitude", score_col].mean()
        ex_alt = sub.loc[sub["altitude_category"] == "Extreme Altitude", score_col].mean()
        n_ex = (sub["altitude_category"] == "Extreme Altitude").sum()
        n_no = (sub["altitude_category"] == "No Altitude").sum()
        if n_ex > 100 and n_no > 100:
            print(f"  {pt:15s}: Extreme={ex_alt:.2f} | No={no_alt:.2f} | Delta={ex_alt - no_alt:+.2f}  (n_ex={n_ex}, n_no={n_no})")



Delta STUFF+ FISICO PURO por tipo de pitch (Extreme vs No Altitude):
  Changeup       : Extreme=96.10 | No=101.35 | Delta=-5.25  (n_ex=19084, n_no=41651)
  Slider         : Extreme=97.99 | No=100.97 | Delta=-2.98  (n_ex=42223, n_no=59821)
  Curveball      : Extreme=97.75 | No=102.98 | Delta=-5.23  (n_ex=14254, n_no=25321)
  Four-Seam      : Extreme=97.10 | No=104.46 | Delta=-7.36  (n_ex=61242, n_no=99566)
  Sinker         : Extreme=95.47 | No=101.58 | Delta=-6.11  (n_ex=31388, n_no=55006)
  Cutter         : Extreme=96.19 | No=100.33 | Delta=-4.14  (n_ex=10002, n_no=9644)
  Splitter       : Extreme=101.53 | No=105.77 | Delta=-4.24  (n_ex=2192, n_no=3534)

Delta PITCHING+ COMPLETO por tipo de pitch (Extreme vs No Altitude):
  Changeup       : Extreme=99.14 | No=100.34 | Delta=-1.20  (n_ex=19084, n_no=41651)
  Slider         : Extreme=99.43 | No=100.28 | Delta=-0.85  (n_ex=42223, n_no=59821)
  Curveball      : Extreme=99.55 | No=100.81 | Delta=-1.25  (n_ex=14254, n_no=25321)
  Four-Seam 

## 12. Test dentro-de-pitcher + Stuff+/Pitching+ final ajustado (entregable)

In [22]:
def within_pitcher_delta(df_eval, score_col, label):
    pitcher_alt = (
        df_eval.groupby(["pitcher_anon_id", "altitude_category"])[score_col]
        .agg(["mean", "size"]).unstack("altitude_category")
    )
    pitcher_alt.columns = ["_".join(map(str, c)) for c in pitcher_alt.columns]
    mask = (
        pitcher_alt["mean_Extreme Altitude"].notna() & pitcher_alt["mean_No Altitude"].notna() &
        (pitcher_alt["size_Extreme Altitude"] >= 50) & (pitcher_alt["size_No Altitude"] >= 50)
    )
    sub = pitcher_alt[mask].copy()
    sub["delta"] = sub["mean_Extreme Altitude"] - sub["mean_No Altitude"]
    print(f"[{label}] n={len(sub)} | Delta medio={sub['delta'].mean():+.2f} | "
          f"% mejoran={((sub['delta']>0).mean()*100):.1f}%")
    return pitcher_alt, sub

pitcher_alt_pure, sub_pure = within_pitcher_delta(df_eval, "Stuff_plus_pure", "Stuff+ fisico puro")
pitcher_alt_pitch, sub_pitch = within_pitcher_delta(df_eval, "Pitching_plus_raw", "Pitching+ completo")

def delta_vs_no(pitcher_alt, cat):
    m = (pitcher_alt[f"mean_{cat}"].notna() & pitcher_alt["mean_No Altitude"].notna() &
         (pitcher_alt[f"size_{cat}"] >= 50) & (pitcher_alt["size_No Altitude"] >= 50))
    sub = pitcher_alt[m].copy()
    sub["d"] = sub[f"mean_{cat}"] - sub["mean_No Altitude"]
    return sub["d"].mean(), len(sub)

d_med, n_med = delta_vs_no(pitcher_alt_pitch, "Medium Altitude")
d_ext, n_ext = delta_vs_no(pitcher_alt_pitch, "Extreme Altitude")
alt_penalty = {"No Altitude": 0.0, "Medium Altitude": d_med, "Extreme Altitude": d_ext}
df_eval["alt_adjustment"] = df_eval["altitude_category"].map(alt_penalty).fillna(0)
df_eval["Pitching_plus_adj"] = df_eval["Pitching_plus_raw"] - df_eval["alt_adjustment"]

pitcher_final = (
    df_eval.groupby("pitcher_anon_id")
    .agg(n=("Pitching_plus_adj", "size"), stuff_pure=("Stuff_plus_pure", "mean"),
         pitching_raw=("Pitching_plus_raw", "mean"), pitching_adj=("Pitching_plus_adj", "mean"),
         pct_altitude=("altitude_ord", lambda x: (x == 2).mean()))
    .query("n >= 200").sort_values("pitching_adj", ascending=False)
)
print(f"\nTop 15 pitchers por Pitching+ ajustado:")
print(pitcher_final.head(15).round(2).to_string())


[Stuff+ fisico puro] n=573 | Delta medio=-5.57 | % mejoran=1.0%
[Pitching+ completo] n=573 | Delta medio=-1.58 | % mejoran=13.3%

Top 15 pitchers por Pitching+ ajustado:
                   n  stuff_pure  pitching_raw  pitching_adj  pct_altitude
pitcher_anon_id                                                           
pitcher_00107    271  105.400002    103.489998        103.91          0.10
pitcher_00172    325  103.529999    102.180000        103.36          0.70
pitcher_00297    357  105.580002    102.940002        103.27          0.19
pitcher_00120    267   99.699997    102.110001        103.23          0.54
pitcher_00626    641  106.570000    102.739998        103.05          0.04
pitcher_00799    220  105.699997    102.529999        103.02          0.31
pitcher_00204    998  105.879997    102.260002        102.98          0.26
pitcher_00759    770  101.410004    101.830002        102.94          0.65
pitcher_01118    388  105.050003    102.110001        102.93          0.14
pitch

In [ ]:
# ============================================================
# Tabla de correlacion: Pitching+ vs. probabilidades promedio por pitcher
# ============================================================
prob_by_pitcher = pitcher_final[["pitching_raw", "pitching_adj"]].copy()

for t, (clf, train_mask, test_mask) in clf_models.items():
    df_test_t = df_test.loc[test_mask]
    X_test_t = X_full.loc[df_test_t.index]
    p = clf.predict_proba(X_test_t)[:, 1]

    prob_series = (
        pd.Series(p, index=df_test_t.index)
        .groupby(df_test_t["pitcher_anon_id"])
        .mean()
        .rename(f"avg_p_{t}")
    )
    prob_by_pitcher = prob_by_pitcher.join(prob_series, how="left")

prob_by_pitcher = prob_by_pitcher.dropna()
print(f"Pitchers con probabilidad promedio calculada: {len(prob_by_pitcher)}")

print(f"\n{'Target':22s} {'Spearman vs Pitching+ raw':>26s}")
print("-" * 50)
for t in clf_models.keys():
    col = f"avg_p_{t}"
    if col in prob_by_pitcher.columns:
        rho, p_val = spearmanr(prob_by_pitcher[col], prob_by_pitcher["pitching_raw"])
        print(f"{t:22s} {rho:>+.4f}  (p={p_val:.2e})")

print("\nDireccion esperada (recuerda: Pitching+ mas ALTO = mejor pitcher):")
print("  target_whiff, target_chase, target_strike, target_out_pa -> correlacion POSITIVA esperada")
print("  target_hard_contact, target_barrel -> signo ambiguo (ver nota abajo)")
print("  target_weak_contact, target_groundball -> signo ambiguo (dependen de contexto/estilo)")

In [ ]:
#Que extraño, groundballs con correlacion negativa? 
# Correlacion groundball vs Pitching+, SOLO en Extreme Altitude
mask_extreme = df_test["altitude_category"] == "Extreme Altitude"
df_test_ext = df_test[mask_extreme]
clf_gb, train_mask_gb, test_mask_gb = clf_models["target_groundball"]
X_ext = X_full.loc[df_test_ext[df_test_ext["is_batted"]==1].index]
df_ext_batted = df_test_ext[df_test_ext["is_batted"]==1]
p_gb_ext = clf_gb.predict_proba(X_ext)[:, 1]

prob_gb_ext = pd.Series(p_gb_ext, index=df_ext_batted.index).groupby(df_ext_batted["pitcher_anon_id"]).mean()
merged_ext = prob_gb_ext.to_frame("p_gb_extreme").join(pitcher_final["pitching_raw"], how="inner")
if len(merged_ext) > 10:
    rho_ext, pval_ext = spearmanr(merged_ext["p_gb_extreme"], merged_ext["pitching_raw"])
    print(f"target_groundball (solo Extreme Altitude) vs Pitching+: {rho_ext:+.4f}  (p={pval_ext:.2e})  n={len(merged_ext)}")

In [ ]:
#Target pa out es raro, no hay correlacion con Pitching+? Será como definimos un pitch?
# Diagnostico rapido: correlacion usando SOLO pitches terminales (is_pa_end==1)
df_test_terminal = df_test[df_test["is_pa_end"] == 1]
clf_outpa, train_mask_outpa, test_mask_outpa = clf_models["target_out_pa"]
X_terminal = X_full.loc[df_test_terminal.index]
p_terminal = clf_outpa.predict_proba(X_terminal)[:, 1]

prob_outpa_terminal = (
    pd.Series(p_terminal, index=df_test_terminal.index)
    .groupby(df_test_terminal["pitcher_anon_id"]).mean()
)
merged = prob_outpa_terminal.to_frame("p_out_terminal").join(pitcher_final["pitching_raw"], how="inner")
rho, pval = spearmanr(merged["p_out_terminal"], merged["pitching_raw"])
print(f"target_out_pa (solo pitches terminales) vs Pitching+: {rho:+.4f}  (p={pval:.2e})")

"La correlación entre P(out en PA) y el score Pitching+ es débil y no significativa a nivel de pitcher (ρ=+0.057, p=0.13) al restringir a pitches terminales. El signo es consistente con lo esperado, pero el tamaño de muestra por pitcher tras el filtro, combinado con la naturaleza ruidosa de los outs en jugada (dependientes de defensa y suerte de contacto), limita la señal detectable en este corte de datos."

# PARTE E · Perfil ideal, estabilidad y selección del score
Sección 8 (perfil) y Sección 5 (estabilidad). SHAP, comparación de cuartiles, persistencia entre temporadas y evaluación a nivel de lanzador.

## 13. Perfil ideal del pitcher Diablos Rojos — SHAP + comparacion de cuartiles
1. **SHAP** sobre el modelo Pitching+ completo: que features pesan mas.
2. **Comparacion de cuartiles**: valores promedio de cada feature fisica/
   arsenal entre el 25% de pitches con MEJOR xRunValue predicho (mas
   favorable al pitcher) vs el 25% con PEOR — da un perfil concreto posible de usar para scouting.

In [ ]:
# --- 1. SHAP sobre el modelo Pitching+ completo (res_full) ---
sample_idx = X_test_full.sample(min(5000, len(X_test_full)), random_state=RANDOM_STATE).index
explainer = shap.TreeExplainer(res_full["model"])
shap_values = explainer.shap_values(X_test_full.loc[sample_idx])

shap_importance = (
    pd.DataFrame({"feature": X_test_full.columns, "mean_abs_shap": np.abs(shap_values).mean(axis=0)})
    .sort_values("mean_abs_shap", ascending=False)
)
print("Top 20 features por importancia SHAP (Pitching+ completo):")
print(shap_importance.head(20).to_string(index=False))


In [ ]:
# --- 2. Comparacion de cuartiles: mejores vs peores pitches predichos ---
df_test_eval = df_test.copy()
df_test_eval["pred_xrv"] = res_full["pred"]

q25 = df_test_eval["pred_xrv"].quantile(0.25)
q75 = df_test_eval["pred_xrv"].quantile(0.75)
best_pitches = df_test_eval[df_test_eval["pred_xrv"] <= q25]   # menor xRunValue predicho = mejor para el pitcher
worst_pitches = df_test_eval[df_test_eval["pred_xrv"] >= q75]

profile_features = stuff_features + arsenal_features
profile_features = [f for f in profile_features if f in df_test_eval.columns]

rows = []
for f in profile_features:
    b, w = best_pitches[f].mean(), worst_pitches[f].mean()
    rows.append({"feature": f, "mejor_25%": b, "peor_25%": w, "diferencia": b - w})

profile_df = pd.DataFrame(rows).sort_values("diferencia", key=lambda s: s.abs(), ascending=False)
print(f"\nPerfil ideal (mejor 25%% de pitches por xRunValue predicho, n={len(best_pitches)}) "
      f"vs peor 25%% (n={len(worst_pitches)}):")
print(profile_df.round(3).to_string(index=False))
print("\nNota: 'mejor 25%' = pitches con menor xRunValue predicho (mas favorables al pitcher).")
print("Estos valores son el punto de partida para un 'perfil fisico ideal' de scouting,")
print("pero recuerda que reflejan correlacion dentro del modelo, no causalidad aislada por feature.")


## 14. Stickiness (correcto: R2 por feature individual, sin artefacto de escalas mezcladas)

In [ ]:
stuff_profile_features = [
    "RelSpeed", "EffectiveVelo", "SpinRate", "SpinAxis", "RelHeight", "RelSide",
    "Extension", "VertBreak", "InducedVertBreak", "HorzBreak",
    "VertRelAngle", "HorzRelAngle", "VertApprAngle", "HorzApprAngle", "ZoneTime",
]

score_by_pitcher_year = (
    df_eval.groupby(["pitcher_anon_id", "year"])[["Stuff_plus_pure", "Pitching_plus_raw"]].agg(["mean", "size"])
)
score_by_pitcher_year.columns = ["_".join(c) for c in score_by_pitcher_year.columns]
score_by_pitcher_year = score_by_pitcher_year.reset_index()

years_sorted = sorted(score_by_pitcher_year["year"].unique())
print("Stickiness de los SCORES COMPUESTOS (ano a ano):")
for y1, y2 in zip(years_sorted[:-1], years_sorted[1:]):
    s1 = score_by_pitcher_year[score_by_pitcher_year["year"] == y1]
    s2 = score_by_pitcher_year[score_by_pitcher_year["year"] == y2]
    merged = s1.merge(s2, on="pitcher_anon_id", suffixes=("_y1", "_y2"))
    merged = merged[(merged["Stuff_plus_pure_size_y1"] >= 100) & (merged["Stuff_plus_pure_size_y2"] >= 100)]
    if len(merged) > 10:
        r_pure, _ = pearsonr(merged["Stuff_plus_pure_mean_y1"], merged["Stuff_plus_pure_mean_y2"])
        r_pitch, _ = pearsonr(merged["Pitching_plus_raw_mean_y1"], merged["Pitching_plus_raw_mean_y2"])
        print(f"  {y1} -> {y2}: Stuff+ puro R2={r_pure**2:.4f} | Pitching+ R2={r_pitch**2:.4f}  (n={len(merged)})")

pitcher_season_stuff = df_cleaned.groupby(["pitcher_anon_id", "year"])[stuff_profile_features].mean().reset_index()

def stickiness_por_feature(df_profiles, y1, y2):
    s1 = df_profiles[df_profiles["year"] == y1]
    s2 = df_profiles[df_profiles["year"] == y2]
    merged = s1.merge(s2, on="pitcher_anon_id", suffixes=("_prev", "_next"))
    rows = []
    for feat in stuff_profile_features:
        x, y = merged[f"{feat}_prev"], merged[f"{feat}_next"]
        mask = x.notna() & y.notna()
        if mask.sum() < 10:
            continue
        r = np.corrcoef(x[mask], y[mask])[0, 1]
        rows.append({"feature": feat, "R2": r ** 2, "n_pitchers": mask.sum()})
    return pd.DataFrame(rows).sort_values("R2", ascending=False)

print("\nStickiness por feature fisica individual:")
for y1, y2 in zip(years_sorted[:-1], years_sorted[1:]):
    res = stickiness_por_feature(pitcher_season_stuff, y1, y2)
    print(f"\n{y1} -> {y2}  (mediana R2 = {res['R2'].median():.4f})")


### Selección del score principal

Se evaluaron dos estrategias para construir un score global de calidad del
lanzamiento:

1. Un composite de probabilidades de los distintos targets, ponderado según
   su relación con run value.
2. Una regresión directa sobre xRunValue.

El composite probabilístico presentó un RMSE superior y una menor estabilidad
del ranking, evaluada mediante correlación de Spearman. Por ello, se seleccionó
la regresión directa sobre xRunValue como score principal.

Los modelos probabilísticos se mantienen como modelos auxiliares para
caracterizar dimensiones específicas del rendimiento del lanzamiento
(whiff, chase, called strike, contacto débil/duro, barrel, groundball y out
within PA).

Por tanto, el score principal utilizado en el análisis es:

xRunValue → XGBoost Regressor → Pitching+

donde Pitching+ se normaliza alrededor de 100, de forma que valores superiores
representan lanzamientos con menor xRunValue predicho y, por tanto, mejor
resultado esperado para el pitcher.

In [ ]:
# ============================================================
# Evaluación a nivel de pitcher: predicciones vs xRunValue real
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

MIN_PITCHES = 200

# --- 1. Predicciones pitch-level sobre el TEST set ---
df_eval_p = df_test.copy()

# Predicciones ya guardadas por evaluate_regression (Sección 8)
# res_full["pred"]  → pred sobre X_test_full (mismo orden que df_test)
# res_stuff["pred"] → pred sobre X_test_stuff (mismo orden que df_test)
df_eval_p["pred_pitching"] = res_full["pred"]
df_eval_p["pred_stuff"]    = res_stuff["pred"]

# --- 2. Agregación por pitcher ---
pitcher_eval = (
    df_eval_p.groupby("pitcher_anon_id")
    .agg(
        n_pitches    = ("pred_pitching", "size"),
        pred_pitching= ("pred_pitching", "mean"),
        pred_stuff   = ("pred_stuff",    "mean"),
        actual_xrv   = ("xRunValue",     "mean"),
    )
    .query("n_pitches >= @MIN_PITCHES")
    .sort_values("pred_pitching", ascending=True)   # más negativo = mejor
)

print(f"Pitchers evaluados: {len(pitcher_eval)}")

# --- 3. Spearman a nivel de pitcher ---
rho_p, p_p = spearmanr(pitcher_eval["pred_pitching"], pitcher_eval["actual_xrv"])
rho_s, p_s = spearmanr(pitcher_eval["pred_stuff"],   pitcher_eval["actual_xrv"])

print(f"Pitching+ Spearman vs xRunValue real: {rho_p:+.4f} (p={p_p:.2e})")
print(f"Stuff+ Spearman vs xRunValue real:    {rho_s:+.4f} (p={p_s:.2e})")

# --- 4. Rankings ---
print("\nRanking de pitchers (pred_pitching, más negativo = mejor):")
print(pitcher_eval.head(10).round(6).to_string())

print("\nPeores pitchers por Pitching+:")
print(
    pitcher_eval
    .sort_values("pred_pitching", ascending=False)
    .head(10)
    .round(6)
    .to_string()
)

# PARTE F · Exportación de modelos y datos del dashboard
Los modelos se guardan como boosters nativos de XGBoost (JSON). El dashboard final no ejecuta modelos en el navegador: usa una **rejilla de predicciones precalculada** (`prediction_grid_v4.json`, bloque F.5).

### F.1 Instalación de librerías 

In [ ]:
!pip install onnxmltools skl2onnx onnxconverter-common

In [ ]:
import numpy as np
print(np.__version__)

In [ ]:
# Ejecutar en Anaconda Prompt (no es código Python):
# conda install -n base numpy=1.26.4 -y


In [ ]:
import numpy as np
print(np.__version__)

### F.2 Modelo de Pitching+ y archivos auxiliares exportado

In [ ]:
import itertools, json

grid = []
for pt, speed, ivb, hb, locH, locS, alt in itertools.product(
    ["Four-Seam","Sinker","Slider","Curveball","Changeup","Cutter","Splitter"],
    [78,85,90,93,96], [0,6,12,18], [-10,0,10], [1.5,2.5,3.5], [-1,0,1], [0,1,2]
):
    row = {...}  # construir la fila con las columnas exactas de X_full, LHP ya normalizado, dummies correctas
    pred = res_full["model"].predict(pd.DataFrame([row]))[0]
    grid.append({"pitch_type":pt,"speed":speed,"ivb":ivb,"hb":hb,"locH":locH,"locS":locS,
                 "alt":alt,"xrv_pred":float(pred)})

json.dump(grid, open("prediction_grid.json","w"))

In [ ]:
import os
OUT_DIR = r"C:\Users\jorge"          # aquí es donde diablos.py lo busca
os.makedirs(OUT_DIR, exist_ok=True)

model_path = os.path.join(OUT_DIR, "pitching_plus_model.json")
res_full["model"].save_model(model_path)

print("Guardado en:", model_path)
print("Existe:", os.path.exists(model_path))
print("Tamaño:", os.path.getsize(model_path), "bytes")
print("N features:", X_full.shape[1])

In [ ]:
import json
with open(r"C:\Users\jorge\feature_order.json", "w") as f:
    json.dump(list(X_full.columns), f, indent=2)
print("Guardado:", len(X_full.columns), "features")
print(list(X_full.columns))

In [ ]:
print("mu_p:", mu_p, "sd_p:", sd_p)

import json
with open(r"C:\Users\jorge\pitching_plus_scaling.json", "w") as f:
    json.dump({"mu": float(mu_p), "sd": float(sd_p)}, f, indent=2)
print("Guardado pitching_plus_scaling.json")

In [ ]:
print(X_full.shape)

In [ ]:
import json
with open(r"C:\Users\jorge\feature_order.json", "w") as f:
    json.dump(list(X_full.columns), f, indent=2)
print("Guardado. Total features:", len(X_full.columns))
print(list(X_full.columns))

In [ ]:
# Exportar 5 filas del test set para probar el .onnx
X_test_full.head(5).to_csv(r"C:\Users\jorge\X_sample.csv", index=False)
print("Guardado:", X_test_full.head(5).shape)
print(X_test_full.head(5).iloc[0][:5])  # primeras 5 features de la primera fila

### F.3 Exportar los 8 clasificadores (boosters nativos) y archivos de verificación

In [ ]:
import json
import numpy as np

# --- 1. Guardar cada uno de los 8 boosters nativos a disco ---
targets = ["target_whiff", "target_chase", "target_strike", "target_out_pa",
           "target_weak_contact", "target_hard_contact", "target_barrel", "target_groundball"]

for t in targets:
    clf, train_mask, test_mask = clf_models[t]
    clf.get_booster().save_model(f"{t}_model.json")
    print(f"Guardado: {t}_model.json")

# --- 2. feature_order ya deberías tenerlo de Pitching+; confirma que es el mismo para los 8 ---
feature_order = list(X_full.columns)
print("feature_order coincide con Pitching+:", feature_order == json.load(open("feature_order.json")))
json.dump(feature_order, open("feature_order.json", "w"))  # sobreescribe/confirma, son las mismas 49 cols

# --- 3. Muestra de sanidad: 5 filas de X_full + predicciones nativas de CADA clasificador ---
X_sample = X_full.iloc[:5]
X_sample.to_csv("sanity_sample_X.csv", index=False)

native_preds = {}
for t in targets:
    clf, _, _ = clf_models[t]
    native_preds[t] = clf.predict_proba(X_sample)[:, 1].tolist()

json.dump(native_preds, open("sanity_native_preds.json", "w"))
print("Listo: 8 model.json + feature_order.json + sanity_sample_X.csv + sanity_native_preds.json")

In [ ]:
import json

# --- 1. Guardar cada uno de los 8 boosters nativos a disco ---
targets = ["target_whiff", "target_chase", "target_strike", "target_out_pa",
           "target_weak_contact", "target_hard_contact", "target_barrel", "target_groundball"]

for t in targets:
    clf, train_mask, test_mask = clf_models[t]
    clf.get_booster().save_model(f"{t}_model.json")
    print(f"Guardado: {t}_model.json")

# --- 2. feature_order (genera el archivo, sin comparar contra uno que aun no existe aqui) ---
feature_order = list(X_full.columns)
json.dump(feature_order, open("feature_order.json", "w"))
print(f"feature_order.json guardado con {len(feature_order)} columnas")

# --- 3. Muestra de sanidad ---
X_sample = X_full.iloc[:5]
X_sample.to_csv("sanity_sample_X.csv", index=False)

native_preds = {}
for t in targets:
    clf, _, _ = clf_models[t]
    native_preds[t] = clf.predict_proba(X_sample)[:, 1].tolist()

json.dump(native_preds, open("sanity_native_preds.json", "w"))
print("Listo: 8 model.json + feature_order.json + sanity_sample_X.csv + sanity_native_preds.json")


In [ ]:
import os
print("Directorio de trabajo del notebook:", os.getcwd())


In [ ]:
import shutil, os

destino = r"C:\Users\jorge"

archivos = [
    "target_whiff_model.json", "target_chase_model.json", "target_strike_model.json",
    "target_out_pa_model.json", "target_weak_contact_model.json",
    "target_hard_contact_model.json", "target_barrel_model.json",
    "target_groundball_model.json",
    "feature_order.json", "sanity_sample_X.csv", "sanity_native_preds.json",
]

for f in archivos:
    origen = os.path.join(os.getcwd(), f)
    shutil.copy2(origen, os.path.join(destino, f))
    print(f"Copiado: {f}")

print(f"\nTodos los archivos copiados a {destino}")

### F.4 Exportar Stuff+ (modelo, orden de variables, escala y muestra de verificación)

In [ ]:
import json

res_stuff["model"].get_booster().save_model("stuff_plus_model.json")

stuff_feature_order = list(X_stuff.columns)
json.dump(stuff_feature_order, open("stuff_feature_order.json", "w"))
print(f"stuff_feature_order.json guardado con {len(stuff_feature_order)} columnas")

mu_s = df_eval["stuff_pred_pure"].mean()
sd_s = df_eval["stuff_pred_pure"].std()
json.dump({"mu": float(mu_s), "sd": float(sd_s)}, open("stuff_plus_scaling.json", "w"))
print(f"mu_s={mu_s}, sd_s={sd_s}")

X_sample_stuff = X_stuff.iloc[:5]
X_sample_stuff.to_csv("sanity_sample_X_stuff.csv", index=False)

native_preds_stuff = res_stuff["model"].predict(X_sample_stuff).tolist()
json.dump(native_preds_stuff, open("sanity_native_preds_stuff.json", "w"))

print("Listo: stuff_plus_model.json + stuff_feature_order.json + stuff_plus_scaling.json + sanity_sample_X_stuff.csv + sanity_native_preds_stuff.json")

In [ ]:
import shutil, os

destino = r"C:\Users\jorge"

archivos = [
    "stuff_plus_model.json", "stuff_feature_order.json", "stuff_plus_scaling.json",
    "sanity_sample_X_stuff.csv", "sanity_native_preds_stuff.json",
]

for f in archivos:
    origen = os.path.join(os.getcwd(), f)
    shutil.copy2(origen, os.path.join(destino, f))
    print(f"Copiado: {f}")

print(f"\nTodos los archivos copiados a {destino}")

### F.5 Rejilla de predicciones del dashboard —  versión final v4

In [ ]:
import json, itertools
import pandas as pd
import numpy as np

# Confirma que tienes mu_p/sd_p (si no los guardaste explícitamente, se recalculan aquí)
mu_p = df_eval["pitching_pred_raw"].mean()
sd_p = df_eval["pitching_pred_raw"].std()

pitch_types = ["Four-Seam","Sinker","Slider","Curveball","Changeup","Cutter","Splitter"]

# Fila base REAL por tipo de pitch (la más cercana a la mediana de velocidad de ese tipo)
base_idx = {}
for pt in pitch_types:
    mask = df_cleaned["AutoPitchType"] == pt
    if mask.sum() == 0: continue
    sub = df_cleaned[mask]
    med = sub["RelSpeed"].median()
    base_idx[pt] = (sub["RelSpeed"] - med).abs().idxmin()

print("Tipos con fila base:", list(base_idx.keys()))

grid_rows = []
for pt, bidx in base_idx.items():
    pt_mask = df_cleaned["AutoPitchType"] == pt
    def lv(col): 
        s = df_cleaned.loc[pt_mask, col]
        return [s.quantile(.15), s.quantile(.5), s.quantile(.85)]
    speed_lv, ivb_lv, hb_lv = lv("RelSpeed"), lv("InducedVertBreak"), lv("HorzBreak")
    locH_lv, locS_lv = [1.8, 2.5, 3.2], [-0.7, 0.0, 0.7]

    base_stuff = X_stuff.loc[bidx].copy()
    base_full = X_full.loc[bidx].copy()

    for speed, ivb, hb, locH, locS, alt in itertools.product(speed_lv, ivb_lv, hb_lv, locH_lv, locS_lv, [0,1,2]):
        rs = base_stuff.copy()
        rs["RelSpeed"], rs["InducedVertBreak"], rs["HorzBreak"] = speed, ivb, hb

        rf = base_full.copy()
        rf["RelSpeed"], rf["InducedVertBreak"], rf["HorzBreak"] = speed, ivb, hb
        rf["PlateLocHeight"], rf["PlateLocSide"], rf["altitude_ord"] = locH, locS, alt
        if "in_strike_zone" in rf.index:
            rf["in_strike_zone"] = int(abs(locS) < 0.83 and 1.5 < locH < 3.6)

        sp = 100 - 10*(res_stuff["model"].predict(pd.DataFrame([rs]))[0] - mu_s) / sd_s
        pp = 100 - 10*(res_full["model"].predict(pd.DataFrame([rf]))[0] - mu_p) / sd_p
        probs = {t: float(clf.predict_proba(pd.DataFrame([rf]))[0,1]) for t,(clf,_,_) in clf_models.items()}

        grid_rows.append({"pitch_type":pt,"speed":round(float(speed),1),"ivb":round(float(ivb),1),
                           "hb":round(float(hb),1),"locH":locH,"locS":locS,"alt":alt,
                           "stuff_plus":round(float(sp),2),"pitching_plus":round(float(pp),2),
                           **{k:round(v,4) for k,v in probs.items()}})

print(f"Grid generado: {len(grid_rows)} filas")
json.dump(grid_rows, open("prediction_grid.json","w"))

*Movimiento real promedio por tipo de pitch y altitud (primera variante del cálculo).*

In [ ]:
real_movement = (
    df_cleaned.groupby(["AutoPitchType","altitude_category"])[["InducedVertBreak","HorzBreak"]]
    .mean().reset_index()
)
pivot_ivb = real_movement.pivot(index="AutoPitchType", columns="altitude_category", values="InducedVertBreak")
pivot_hb = real_movement.pivot(index="AutoPitchType", columns="altitude_category", values="HorzBreak")

print("Reducción real de IVB (% vs No Altitude):")
print(((pivot_ivb["Extreme Altitude"] - pivot_ivb["No Altitude"]) / pivot_ivb["No Altitude"].abs() * 100).round(1))
print("\nReducción real de HB (% vs No Altitude):")
print(((pivot_hb["Extreme Altitude"] - pivot_hb["No Altitude"]) / pivot_hb["No Altitude"].abs() * 100).round(1))

*Movimiento real promedio por tipo de pitch y altitud (segunda variante, con porcentajes guardados).*

In [ ]:
real_movement = (
    df_cleaned.groupby(["AutoPitchType","altitude_category"])[["InducedVertBreak","HorzBreak"]]
    .mean().reset_index()
)
pivot_ivb = real_movement.pivot(index="AutoPitchType", columns="altitude_category", values="InducedVertBreak")
pivot_hb = real_movement.pivot(index="AutoPitchType", columns="altitude_category", values="HorzBreak")

ivb_pct = ((pivot_ivb["Extreme Altitude"] - pivot_ivb["No Altitude"]) / pivot_ivb["No Altitude"].abs() * 100).round(1)
hb_pct = ((pivot_hb["Extreme Altitude"] - pivot_hb["No Altitude"]) / pivot_hb["No Altitude"].abs() * 100).round(1)
print("Reducción real IVB (%):\n", ivb_pct)
print("\nReducción real HB (%):\n", hb_pct)

### F.6 Movimiento (IVB/HB) dentro de lanzador y verificación de que Stuff+ responde a la física

In [ ]:
# IVB/HB dentro-de-pitcher: mismo pitcher, mismo tipo de pitch, distinta altitud
movement_by_pitcher = (
    df_cleaned.groupby(["pitcher_anon_id", "AutoPitchType", "altitude_category"])
    [["InducedVertBreak", "HorzBreak"]]
    .agg(["mean", "size"])
)
movement_by_pitcher.columns = ["_".join(c) for c in movement_by_pitcher.columns]
movement_by_pitcher = movement_by_pitcher.reset_index()

results = []
for pt in pitch_types:
    sub = movement_by_pitcher[movement_by_pitcher["AutoPitchType"] == pt]
    piv_ivb = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_mean")
    piv_hb = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="HorzBreak_mean")
    piv_n_ex = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_size")["Extreme Altitude"] if "Extreme Altitude" in sub["altitude_category"].values else None

    mask = (
        piv_ivb["Extreme Altitude"].notna() & piv_ivb["No Altitude"].notna() &
        (sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_size")["Extreme Altitude"].reindex(piv_ivb.index) >= 20) &
        (sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_size")["No Altitude"].reindex(piv_ivb.index) >= 20)
    ) if "Extreme Altitude" in piv_ivb.columns and "No Altitude" in piv_ivb.columns else pd.Series(False, index=piv_ivb.index)

    n_pitchers = mask.sum()
    if n_pitchers >= 10:
        ivb_delta_pct = ((piv_ivb.loc[mask,"Extreme Altitude"] - piv_ivb.loc[mask,"No Altitude"]) / piv_ivb.loc[mask,"No Altitude"].abs() * 100).mean()
        hb_delta_pct = ((piv_hb.loc[mask,"Extreme Altitude"] - piv_hb.loc[mask,"No Altitude"]) / piv_hb.loc[mask,"No Altitude"].abs() * 100).mean()
        results.append({"pitch_type": pt, "n_pitchers": n_pitchers, "ivb_delta_pct": round(ivb_delta_pct,1), "hb_delta_pct": round(hb_delta_pct,1)})
    else:
        results.append({"pitch_type": pt, "n_pitchers": n_pitchers, "ivb_delta_pct": None, "hb_delta_pct": None})

pd.DataFrame(results)

In [ ]:
results_abs = []
for pt in pitch_types:
    sub = movement_by_pitcher[movement_by_pitcher["AutoPitchType"] == pt]
    piv_ivb = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_mean")
    piv_hb = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="HorzBreak_mean")
    piv_n_ex = sub.pivot(index="pitcher_anon_id", columns="altitude_category", values="InducedVertBreak_size")

    if "Extreme Altitude" not in piv_ivb.columns or "No Altitude" not in piv_ivb.columns:
        continue
    mask = (
        piv_ivb["Extreme Altitude"].notna() & piv_ivb["No Altitude"].notna() &
        (piv_n_ex["Extreme Altitude"] >= 20) & (piv_n_ex["No Altitude"] >= 20)
    )
    n_pitchers = mask.sum()
    if n_pitchers >= 10:
        ivb_delta_abs = (piv_ivb.loc[mask,"Extreme Altitude"] - piv_ivb.loc[mask,"No Altitude"]).mean()
        hb_delta_abs = (piv_hb.loc[mask,"Extreme Altitude"] - piv_hb.loc[mask,"No Altitude"]).mean()
        results_abs.append({"pitch_type": pt, "n_pitchers": int(n_pitchers),
                             "ivb_delta_in": round(ivb_delta_abs,2), "hb_delta_in": round(hb_delta_abs,2)})

pd.DataFrame(results_abs)

### F.7 Rejilla v4

In [ ]:
import itertools, json, os
import numpy as np
import pandas as pd

K = 15
PITCH_TYPES = ["Four-Seam","Sinker","Slider","Curveball","Changeup","Cutter","Splitter"]
COUNT_STATES = [(0,0),(1,1),(3,2)]
ALTS = [0,1,2]
TARGETS = ["target_whiff","target_chase","target_strike","target_out_pa",
           "target_weak_contact","target_hard_contact","target_barrel","target_groundball"]

assert X_full.index.equals(df_cleaned.index) and X_stuff.index.equals(df_cleaned.index), "índices desalineados"

p_cols = [c for c in X_full.columns if c.startswith("PitcherThrows_")]
b_cols = [c for c in X_full.columns if c.startswith("BatterSide_")]
HAS_HAND = bool(p_cols) and bool(b_cols)
HANDS = ["Right","Left"] if HAS_HAND else [None]
SIDES = ["Right","Left"] if HAS_HAND else [None]
print("Mano en el modelo:", HAS_HAND, p_cols, b_cols)
print("PitcherThrows:", df_cleaned["PitcherThrows"].unique(), "| BatterSide:", df_cleaned["BatterSide"].unique())

# Constantes de normalización con los MISMOS modelos que se muestran
pred_stuff_all = res_stuff["model"].predict(X_stuff)
mu_s, sd_s = float(pred_stuff_all.mean()), float(pred_stuff_all.std())
pred_full_all = res_full["model"].predict(X_full)
mu_p, sd_p = float(pred_full_all.mean()), float(pred_full_all.std())

ptype = df_cleaned["AutoPitchType"].astype(str).to_numpy()
phand = df_cleaned["PitcherThrows"].astype(str).to_numpy()
phys  = df_cleaned[["RelSpeed","InducedVertBreak","HorzBreak"]].to_numpy(dtype=float)
extra = df_cleaned[["RelHeight","ZoneTime","VertRelAngle","VertApprAngle"]].to_numpy(dtype=float)
cols = list(X_full.columns); cidx = {c: i for i, c in enumerate(cols)}
Xf = X_full.to_numpy(dtype=np.float32)
Xs = X_stuff.to_numpy(dtype=np.float32)
scale = phys.std(axis=0)

def quants(pt, col, qs):
    s = df_cleaned.loc[ptype == pt, col]
    return [round(float(s.quantile(q)), 2) for q in qs]

LEVELS = {pt: {"speed": quants(pt,"RelSpeed",(.15,.5,.85)), "ivb": quants(pt,"InducedVertBreak",(.15,.5,.85)),
               "hb": quants(pt,"HorzBreak",(.15,.5,.85)), "locH": quants(pt,"PlateLocHeight",(.10,.5,.90)),
               "locS": quants(pt,"PlateLocSide",(.10,.5,.90))} for pt in PITCH_TYPES}

cells_rows, pred_rows = [], []
cell_ids = list(itertools.product(range(3), range(3), range(3)))     # (vel, IVB, HB)

for ti, pt in enumerate(PITCH_TYPES):
    L = LEVELS[pt]
    targets = np.array([[L["speed"][s], L["ivb"][i], L["hb"][h]] for s, i, h in cell_ids])
    for hi_, H in enumerate(HANDS):
        mask = (ptype == pt) & ((phand == H) if H else True)
        pool = np.where(mask)[0]
        if len(pool) == 0:
            print(f"AVISO: sin pitches de {pt} / {H}; se omite"); continue
        k = min(K, len(pool))
        Z = phys[pool] / scale
        nbr = np.empty((27, k), dtype=int)
        for c, tgt in enumerate(targets):
            d = ((Z - tgt/scale)**2).sum(axis=1)
            nbr[c] = pool[np.argpartition(d, k-1)[:k]]
        flat = nbr.reshape(-1)

        # Stuff+ y valores físicos efectivos por celda
        sp = 100 - 10*(res_stuff["model"].predict(pd.DataFrame(Xs[flat], columns=X_stuff.columns)) - mu_s)/sd_s
        sp = sp.reshape(27, k).mean(axis=1)
        eff = phys[flat].reshape(27, k, 3).mean(axis=1)
        ex = extra[flat].reshape(27, k, 4).mean(axis=1)
        for c, (s, i, h) in enumerate(cell_ids):
            cells_rows.append([ti, hi_, s, i, h] + np.round(eff[c], 2).tolist() + np.round(ex[c], 3).tolist()
                              + [round(float(sp[c]), 2), int(k)])

        # Pitching+ y probabilidades: ubicación x conteo x altitud x lado del bateador
        combos = list(itertools.product(range(3), range(3), ALTS, range(3), SIDES))
        base = Xf[flat]; n_nb = base.shape[0]
        big = np.tile(base, (len(combos), 1))                          # combo-mayor
        def per_combo(fn): return np.repeat(np.array([fn(c) for c in combos], dtype=np.float32), n_nb)
        big[:, cidx["PlateLocHeight"]] = per_combo(lambda c: L["locH"][c[0]])
        big[:, cidx["PlateLocSide"]]   = per_combo(lambda c: L["locS"][c[1]])
        big[:, cidx["altitude_ord"]]   = per_combo(lambda c: c[2])
        big[:, cidx["Balls"]]   = per_combo(lambda c: COUNT_STATES[c[3]][0])
        big[:, cidx["Strikes"]] = per_combo(lambda c: COUNT_STATES[c[3]][1])
        big[:, cidx["Outs"]] = 0.0; big[:, cidx["Inning"]] = 1.0       # fijos: 0 outs, 1a entrada
        if "in_strike_zone" in cidx:
            big[:, cidx["in_strike_zone"]] = per_combo(
                lambda c: float(abs(L["locS"][c[1]]) < 0.83 and 1.5 < L["locH"][c[0]] < 3.6))
        if HAS_HAND:
            for cc in p_cols: big[:, cidx[cc]] = 1.0 if cc == f"PitcherThrows_{H}" else 0.0
            for cc in b_cols: big[:, cidx[cc]] = per_combo(lambda c, cc=cc: 1.0 if cc == f"BatterSide_{c[4]}" else 0.0)

        Xbig = pd.DataFrame(big, columns=cols)
        pp = 100 - 10*(res_full["model"].predict(Xbig) - mu_p)/sd_p
        pr = {t: clf_models[t][0].predict_proba(Xbig)[:, 1] for t in TARGETS}
        avg = lambda a: a.reshape(len(combos), 27, k).mean(axis=2)
        pp_m, pr_m = avg(pp), {t: avg(v) for t, v in pr.items()}
        for j, (li, lj, alt, ci, side) in enumerate(combos):
            si_ = SIDES.index(side) if HAS_HAND else 0
            for c, (s, i, h) in enumerate(cell_ids):
                pred_rows.append([ti, hi_, si_, s, i, h, li, lj, alt, ci, round(float(pp_m[j, c]), 2)]
                                 + [round(float(pr_m[t][j, c]), 4) for t in TARGETS])
    print(f"{pt:10s} listo")

out = {"meta": {"pitch_types": PITCH_TYPES, "hands": [h or "All" for h in HANDS], "sides": [s or "All" for s in SIDES],
                "count_states": COUNT_STATES, "levels": LEVELS, "targets": TARGETS, "has_hand": HAS_HAND, "k": K,
                "mu_p": mu_p, "sd_p": sd_p, "mu_s": mu_s, "sd_s": sd_s},
       "cell_cols": ["pt","hand","si","ii","hi","speed","ivb","hb","rel_h","zone_t","vra","vaa","stuff_plus","k"],
       "cells": cells_rows,
       "pred_cols": ["pt","hand","side","si","ii","hi","li","lj","alt","ci","pitching_plus"] + TARGETS,
       "pred": pred_rows}
json.dump(out, open("prediction_grid_v4.json", "w"), separators=(",", ":"))

exp = len(PITCH_TYPES)*len(HANDS)*27*81*len(SIDES)
print(f"\nfilas pred: {len(pred_rows)} (esperadas {exp}) | celdas: {len(cells_rows)} | "
      f"{os.path.getsize('prediction_grid_v4.json')/1e6:.1f} MB")
pdf = pd.DataFrame(pred_rows, columns=out["pred_cols"])
print(pdf[["pitching_plus"] + TARGETS].describe().loc[["min", "max"]].T.round(3))

# PARTE G · Resultados finales y tablas del informe
Secciones 5 a 8.* Cada bloque genera una o más tablas del informe y, cuando corresponde, el archivo CSV indicado.

### G.1 Scores fuera de muestra (OOF) y normalización por temporada × categoría de altitud (proxy de estadio)

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import GroupKFold

y = df_cleaned["xRunValue"]
groups = df_cleaned["pitcher_anon_id"].fillna("unknown")

# BEST_PARAMS puede traer n_jobs / tree_method / random_state: los tuyos mandan
XGB_KW = {"tree_method": "hist", "n_jobs": -1, "random_state": 42}
XGB_KW.update(BEST_PARAMS)

def oof_pred(X, n_splits=5):
    oof = np.zeros(len(X), dtype="float32")
    for tr, te in GroupKFold(n_splits).split(X, y, groups):
        m = xgb.XGBRegressor(**XGB_KW)
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = m.predict(X.iloc[te])
    return oof

pred_stuff = oof_pred(X_stuff)
pred_loc   = oof_pred(X_loc_only)    # Location+ = ubicación + conteo
pred_full  = oof_pred(X_full)

def plus_global(p):
    p = pd.Series(p, index=df_cleaned.index)
    return 100 - 10 * (p - p.mean()) / p.std()

S = df_cleaned[["PitchUID","year","pitcher_anon_id","AutoPitchType","altitude_category"]].copy()
S["stuff_plus"]    = plus_global(pred_stuff)
S["location_plus"] = plus_global(pred_loc)
S["pitching_plus"] = plus_global(pred_full)

# Normalización por temporada x categoría de altitud (proxy de estadio)
S["_p"] = pred_full
g = S.groupby(["year","altitude_category"], dropna=False)["_p"]
S["pitching_plus_adj"] = 100 - 10 * (S["_p"] - g.transform("mean")) / g.transform("std")
S = S.drop(columns="_p")

print(S[["stuff_plus","location_plus","pitching_plus","pitching_plus_adj"]].describe().round(2))
print(S.groupby("altitude_category")[["pitching_plus","pitching_plus_adj"]].mean().round(2))

### G.2 Resumen por lanzador y por arsenal — `scores_por_pitcher.csv`, `scores_por_pitcheo.csv`

In [ ]:
MIN_P = 300   # pitcheos mínimos por pitcher

# --- por pitcher ---
P = (S.groupby("pitcher_anon_id")
       .agg(n=("stuff_plus","size"), stuff_plus=("stuff_plus","mean"),
            location_plus=("location_plus","mean"), pitching_plus=("pitching_plus","mean"),
            pitching_plus_adj=("pitching_plus_adj","mean"))
       .query("n >= @MIN_P").round(2).sort_values("pitching_plus_adj", ascending=False))
print(P.head(15))

# --- arsenal = tipos con >=10% de uso, ordenados por uso ---
usage = (S.groupby(["pitcher_anon_id","AutoPitchType"], observed=True).size()
           .unstack(fill_value=0))
share = usage.div(usage.sum(axis=1), axis=0)
arsenal_key = share.apply(lambda r: "/".join(r[r >= 0.10].sort_values(ascending=False).index), axis=1)
S["arsenal"] = S["pitcher_anon_id"].map(arsenal_key)

# --- qué arsenal rinde más en altitud extrema ---
ext = S[(S["altitude_category"] == "Extreme Altitude") & S["pitcher_anon_id"].isin(P.index)]
by_p = ext.groupby(["arsenal","pitcher_anon_id"]).pitching_plus.mean().reset_index()
A = (by_p.groupby("arsenal").agg(pitchers=("pitcher_anon_id","nunique"), pitching_plus=("pitching_plus","mean"))
          .query("pitchers >= 3").round(2).sort_values("pitching_plus", ascending=False))
print("\nArsenales en Harp Helú (>=3 pitchers):"); print(A.head(10))

# --- por tipo de pitch: sea level vs extrema, dentro de pitcher ---
w = (S[S.altitude_category.isin(["No Altitude","Extreme Altitude"])]
       .groupby(["pitcher_anon_id","AutoPitchType","altitude_category"], observed=True)
       .agg(n=("pitching_plus","size"), pp=("pitching_plus","mean")).reset_index()
       .query("n >= 30").pivot_table(index=["pitcher_anon_id","AutoPitchType"],
                                     columns="altitude_category", values="pp").dropna())
w["delta"] = w["Extreme Altitude"] - w["No Altitude"]
print("\nΔ Pitching+ dentro de pitcher (Extrema − Sin altitud):")
print(w.groupby("AutoPitchType", observed=True).delta.agg(["mean","count"]).round(2))

# --- export ---
P.to_csv("scores_por_pitcher.csv")
A.to_csv("scores_por_arsenal_harp_helu.csv")
EXPORT_PITCH_LEVEL = False   # ~600k filas; actívalo solo si el brief lo pide
if EXPORT_PITCH_LEVEL:
    S.drop(columns=["p"], errors="ignore").to_csv("scores_por_pitcheo.csv", index=False)

### G.3 Scores neutralizados por conteo y outs (`_cn`) — `scores_por_pitcher_v2.csv`

In [ ]:
import numpy as np, pandas as pd

# 1) Neutralizar por conteo (y outs para Pitching+)
B, St, Ou = df_cleaned["Balls"], df_cleaned["Strikes"], df_cleaned["Outs"]
def neutral(col, keys):
    r = S[col] - S.groupby(keys)[col].transform("mean")
    return 100 + 10 * r / r.std()
S["location_plus_cn"]      = neutral("location_plus",      [B, St])
S["pitching_plus_cn"]      = neutral("pitching_plus",      [B, St, Ou])
S["pitching_plus_adj_cn"]  = neutral("pitching_plus_adj",  [B, St, Ou])

# 2) Ranking por pitcher con encogimiento hacia 100
K = 300
P2 = (S.groupby("pitcher_anon_id")
        .agg(n=("stuff_plus","size"), stuff_plus=("stuff_plus","mean"),
             location_cn=("location_plus_cn","mean"), pitching_adj_cn=("pitching_plus_adj_cn","mean"))
        .query("n >= @MIN_P"))
for c in ["stuff_plus","location_cn","pitching_adj_cn"]:
    P2[c] = 100 + (P2[c] - 100) * P2["n"] / (P2["n"] + K)
P2 = P2.round(2).sort_values("stuff_plus", ascending=False)
print(P2.head(15)); print(P2[["stuff_plus","location_cn","pitching_adj_cn"]].std().round(2))
P2.to_csv("scores_por_pitcher_v2.csv")

# 3) Δ por tipo de pitch con intervalo (usa `w` de la celda B)
d = w.reset_index().groupby("AutoPitchType", observed=True).delta.agg(["mean","count","std"])
d["se"] = d["std"] / np.sqrt(d["count"])
d["lo"], d["hi"] = d["mean"] - 1.96*d["se"], d["mean"] + 1.96*d["se"]
print(d.round(2))

# 4) Arsenales en Harp Helú con bootstrap por pitcher (usa `by_p` de la celda B)
rng = np.random.default_rng(0); base = by_p.pitching_plus.mean(); rows = []
for a, grp in by_p.groupby("arsenal"):
    v = grp.pitching_plus.values
    if len(v) < 3: continue
    bs = [rng.choice(v, len(v)).mean() for _ in range(2000)]
    rows.append((a, len(v), v.mean(), *np.percentile(bs, [2.5, 97.5])))
AR = pd.DataFrame(rows, columns=["arsenal","pitchers","mean","lo","hi"]).round(2).sort_values("mean", ascending=False)
AR["distinto_del_promedio"] = (AR["lo"] > base) | (AR["hi"] < base)
print(f"Promedio en Harp Helú: {base:.2f}"); print(AR.head(10))

### G.4 Δ por altitud dentro de lanzador y tipo, con encogimiento — `delta_altitud_por_pitcher.csv`

In [ ]:
import numpy as np, pandas as pd

MIN_EACH = 100     # pitcheos mínimos del pitcher en cada ambiente
MIN_CELL = 20      # mínimo por (pitcher, tipo) en cada ambiente

sub = S[S.altitude_category.isin(["No Altitude","Extreme Altitude"])].copy()
sub["alt"] = sub.altitude_category.map({"No Altitude":"sea","Extreme Altitude":"ext"})
cell = (sub.groupby(["pitcher_anon_id","AutoPitchType","alt"], observed=True)
           .agg(n=("pitching_plus","size"), pp=("pitching_plus","mean"),
                v=("pitching_plus","var"), sp=("stuff_plus","mean")).reset_index())
wd = cell.pivot(index=["pitcher_anon_id","AutoPitchType"], columns="alt", values=["n","pp","v","sp"]).dropna()
wd = wd[(wd[("n","sea")] >= MIN_CELL) & (wd[("n","ext")] >= MIN_CELL)]

c = pd.DataFrame({
    "n_sea": wd[("n","sea")], "n_ext": wd[("n","ext")],
    "d":  wd[("pp","ext")] - wd[("pp","sea")],
    "dS": wd[("sp","ext")] - wd[("sp","sea")],
    "se2": wd[("v","ext")]/wd[("n","ext")] + wd[("v","sea")]/wd[("n","sea")],
})
c["w"] = 1 / (1/c.n_ext + 1/c.n_sea)
c["wd"], c["wdS"], c["w2se2"] = c.w*c.d, c.w*c.dS, c.w**2 * c.se2

g = c.groupby("pitcher_anon_id")
R = pd.DataFrame({
    "n_sea": g.n_sea.sum(), "n_ext": g.n_ext.sum(),
    "delta_pp":    g.wd.sum()  / g.w.sum(),
    "delta_stuff": g.wdS.sum() / g.w.sum(),
    "se":          np.sqrt(g.w2se2.sum()) / g.w.sum(),
}).query("n_sea >= @MIN_EACH and n_ext >= @MIN_EACH")

# Encogimiento hacia el promedio de liga (los extremos de estimaciones ruidosas regresan a la media)
mu = np.average(R.delta_pp, weights=1/R.se**2)
tau2 = max(R.delta_pp.var() - (R.se**2).mean(), 0.0)
R["delta_shrunk"] = mu + (R.delta_pp - mu) * tau2 / (tau2 + R.se**2)
R["z_vs_liga"] = (R.delta_pp - mu) / R.se
R = R.round(2)

print(f"Pitchers calificados: {len(R)} | Δ promedio de liga: {mu:.2f} | tau (dispersión real entre pitchers): {np.sqrt(tau2):.2f}")
cols = ["n_sea","n_ext","delta_pp","se","delta_shrunk","z_vs_liga","delta_stuff"]
print("\nMENOS afectados / mejoran (Δ Pitching+ extrema − sin altitud, mayor = mejor):")
print(R.sort_values("delta_shrunk", ascending=False)[cols].head(10))
print("\nMÁS afectados / empeoran:")
print(R.sort_values("delta_shrunk")[cols].head(10))
R.sort_values("delta_shrunk", ascending=False).to_csv("delta_altitud_por_pitcher.csv")

### G.5 Cambios físicos (velocidad, IVB, HB, spin) que acompañan al Δ

In [ ]:
feat = ["RelSpeed","InducedVertBreak","HorzBreak","SpinRate"]
dfm = df_cleaned.loc[sub.index, ["pitcher_anon_id","AutoPitchType"]+feat].copy()
dfm["alt"] = sub["alt"]
m  = dfm.groupby(["pitcher_anon_id","AutoPitchType","alt"], observed=True)[feat].mean()
t  = m.reset_index().pivot(index=["pitcher_anon_id","AutoPitchType"], columns="alt", values=feat).dropna()
D  = pd.DataFrame({f"d_{f}": t[(f,"ext")] - t[(f,"sea")] for f in feat}).join(c["w"], how="inner")
Dp = (D.drop(columns="w").mul(D["w"], axis=0).groupby("pitcher_anon_id").sum()
        .div(D.groupby("pitcher_anon_id")["w"].sum(), axis=0))
R2 = R.join(Dp, how="inner")
dc = [f"d_{f}" for f in feat]
print(R2[["delta_pp","delta_stuff"]+dc].corr()["delta_pp"].round(2))
top = R2.nlargest(30, "delta_shrunk")[dc+["delta_stuff"]].mean().round(2)
bot = R2.nsmallest(30, "delta_shrunk")[dc+["delta_stuff"]].mean().round(2)
print(pd.DataFrame({"30 que menos pierden": top, "30 que más pierden": bot}))

### G.6 Regresión del Δ contra el cambio en Location+

In [ ]:
from sklearn.linear_model import LinearRegression

ml = (S.loc[sub.index].assign(alt=sub["alt"])
        .groupby(["pitcher_anon_id","AutoPitchType","alt"], observed=True)["location_plus_cn"].mean()
        .reset_index().pivot(index=["pitcher_anon_id","AutoPitchType"], columns="alt", values="location_plus_cn").dropna())
dl = (ml["ext"] - ml["sea"]).rename("d_loc").to_frame().join(c["w"], how="inner")
dlp = (dl["d_loc"]*dl["w"]).groupby("pitcher_anon_id").sum() / dl.groupby("pitcher_anon_id")["w"].sum()
R3 = R2.join(dlp.rename("d_loc"), how="inner").dropna()

print(R3[["delta_pp","delta_stuff","d_loc"]].corr().round(2))
y = R3["delta_pp"]
for name, cols in [("solo físico", dc), ("solo ubicación", ["d_loc"]), ("físico + ubicación", dc+["d_loc"])]:
    r2 = LinearRegression().fit(R3[cols], y).score(R3[cols], y)
    print(f"R² {name:20s}: {r2:.2f}")

### G.7 Estabilidad del Δ entre mitades de temporada

In [ ]:
import numpy as np, pandas as pd

COL = "pitching_plus_cn"    # neutral al conteo; cambia a "pitching_plus" para comparar

def pitcher_delta(df, min_cell=15, min_each=60):
    cell = (df.groupby(["pitcher_anon_id","AutoPitchType","alt"], observed=True)[COL]
              .agg(["size","mean"]).reset_index())
    wd = cell.pivot(index=["pitcher_anon_id","AutoPitchType"], columns="alt",
                    values=["size","mean"]).dropna()
    wd = wd[(wd[("size","sea")] >= min_cell) & (wd[("size","ext")] >= min_cell)]
    d = wd[("mean","ext")] - wd[("mean","sea")]
    w = 1 / (1/wd[("size","ext")] + 1/wd[("size","sea")])
    out = (pd.DataFrame({"dw": d*w, "w": w, "ne": wd[("size","ext")], "ns": wd[("size","sea")]})
             .groupby("pitcher_anon_id").sum())
    out["delta"] = out["dw"] / out["w"]
    return out.query("ne >= @min_each and ns >= @min_each")["delta"]

years = sorted(sub["year"].unique()); A, B = years[::2], years[1::2]
dA = pitcher_delta(sub[sub["year"].isin(A)]); dB = pitcher_delta(sub[sub["year"].isin(B)])
j = pd.concat([dA.rename("A"), dB.rename("B")], axis=1, join="inner")
print("Temporadas:", years, "| mitad A:", A, "| mitad B:", B)
print(f"Pitchers en ambas mitades: {len(j)} | Δ medio A: {j.A.mean():.2f}, B: {j.B.mean():.2f}")
print(f"Correlación Pearson: {j.corr().iloc[0,1]:.2f} | Spearman: {j.corr(method='spearman').iloc[0,1]:.2f}")

### G.8 Spin real por celda del dashboard — `cells_spin.json`

In [ ]:
import json, numpy as np

g = json.load(open("prediction_grid_v4.json")); meta, cc, cells = g["meta"], g["cell_cols"], g["cells"]
ix = {c:i for i,c in enumerate(cc)}
F = {"speed":"RelSpeed", "ivb":"InducedVertBreak", "hb":"HorzBreak"}
groups = {k: d for k, d in df_cleaned.groupby(["AutoPitchType","PitcherThrows"], observed=True)}
sd = {k: {f: d[F[f]].std() for f in F} for k, d in groups.items()}

spin, err = [], []
for r in cells:
    pt, hand = meta["pitch_types"][r[ix["pt"]]], meta["hands"][r[ix["hand"]]]
    L = meta["levels"][pt]
    tgt = {"speed": L["speed"][r[ix["si"]]], "ivb": L["ivb"][r[ix["ii"]]], "hb": L["hb"][r[ix["hb"] if False else ix["hi"]]]}
    d = groups[(pt, hand)]
    z = sum(((d[F[f]] - tgt[f]) / sd[(pt,hand)][f])**2 for f in F)
    nn = d.loc[z.nsmallest(15).index]
    spin.append(round(float(nn["SpinRate"].mean()), 0))
    err.append(max(abs(nn[F["speed"]].mean()-r[ix["speed"]]), abs(nn[F["ivb"]].mean()-r[ix["ivb"]]), abs(nn[F["hb"]].mean()-r[ix["hb"]])))
print("Error máx. vs. celdas originales (mph/pulgadas):", round(max(err),2), "| mediana:", round(float(np.median(err)),2))
json.dump(spin, open("cells_spin.json","w"))

In [ ]:
spin_out = [s if e <= 0.3 else None for s, e in zip(spin, err)]
print("celdas con spin válido:", sum(x is not None for x in spin_out), "de", len(spin_out))
json.dump(spin_out, open("cells_spin.json", "w"))

### G.9 Perfil físico ideal por tipo de pitch y SHAP agrupado por capa — `perfil_ideal_*.csv`

In [ ]:
import numpy as np, pandas as pd, shap

# ---------- 1) Perfil fisico del cuartil superior de Stuff+ por tipo de pitch ----------
feats = {"RelSpeed":"Velocidad (mph)", "InducedVertBreak":"IVB (in)", "HorzBreak":"HB (in)",
         "SpinRate":"Spin (rpm)", "VertApprAngle":"VAA (°)", "Extension":"Extensión (ft)",
         "RelHeight":"Altura lib. (ft)"}
D = df_cleaned[list(feats) + ["AutoPitchType","altitude_category"]].copy()
D["stuff_plus"] = S["stuff_plus"]
# cuartil superior DENTRO de cada tipo de pitch (compara un slider contra sliders)
D["top25"] = D.groupby("AutoPitchType", observed=True)["stuff_plus"].transform(lambda s: s >= s.quantile(0.75))

n_tipo = D.groupby("AutoPitchType", observed=True).size()
keep = n_tipo[n_tipo >= 1000].index
Dk = D[D["AutoPitchType"].isin(keep)]
g = lambda m: Dk[m].groupby("AutoPitchType", observed=True)[list(feats)].mean()

top, rest = g(Dk.top25), g(~Dk.top25)
perfil = top.rename(columns=feats).round(1)
perfil.insert(0, "n_top25", Dk[Dk.top25].groupby("AutoPitchType", observed=True).size())
print("PERFIL MEDIO DEL CUARTIL SUPERIOR DE STUFF+ (todas las altitudes):"); print(perfil)
print("\nDIFERENCIA vs. el resto de los pitcheos del mismo tipo:")
print((top - rest).rename(columns=feats).round(1))

# ---------- 1b) Mismo perfil en altitud extrema vs. sin altitud (como cambia 'ideal'?) ----------
ext = Dk[Dk.top25 & (Dk.altitude_category == "Extreme Altitude")].groupby("AutoPitchType", observed=True)[list(feats)].mean()
sea = Dk[Dk.top25 & (Dk.altitude_category == "No Altitude")].groupby("AutoPitchType", observed=True)[list(feats)].mean()
print("\nCAMBIO del perfil del cuartil superior: altitud extrema − sin altitud:")
print((ext - sea).rename(columns=feats).round(1))

perfil.to_csv("perfil_ideal_por_tipo.csv")
(ext - sea).rename(columns=feats).round(2).to_csv("perfil_ideal_cambio_altitud.csv")

# ---------- 2) SHAP del modelo de Pitching+ ----------
model = res_full["model"]
rng = np.random.default_rng(0)
idx = rng.choice(len(X_full), size=20000, replace=False)
Xs = X_full.iloc[idx]
sv = shap.TreeExplainer(model).shap_values(Xs)

imp = pd.DataFrame({"mean_abs_shap": np.abs(sv).mean(axis=0)}, index=Xs.columns)
# signo: + = valores altos de la variable suben el xRunValue (peor para el lanzador)
imp["signo_sobre_xRV"] = [np.corrcoef(Xs[c], sv[:, i])[0, 1] if Xs[c].std() > 0 else np.nan
                          for i, c in enumerate(Xs.columns)]
imp["pct_total"] = 100 * imp["mean_abs_shap"] / imp["mean_abs_shap"].sum()
imp = imp.sort_values("mean_abs_shap", ascending=False)
print("\nTOP 15 VARIABLES (SHAP, Pitching+):"); print(imp.head(15).round(4))

# peso por capa
stuff_cols = set(X_stuff.columns)
loc_cols = {"PlateLocHeight","PlateLocSide","in_strike_zone","Balls","Strikes"}
ars_cols = set(X_stuff_loc_ars.columns) - stuff_cols - loc_cols
def capa(c):
    return "Stuff" if c in stuff_cols else "Ubicación y conteo" if c in loc_cols else "Arsenal" if c in ars_cols else "Contexto, mano y altitud"
imp["capa"] = [capa(c) for c in imp.index]
por_capa = imp.groupby("capa")["pct_total"].sum().sort_values(ascending=False).round(1)
print("\nPESO POR CAPA (% del SHAP total):"); print(por_capa)
imp.round(5).to_csv("shap_pitching_plus.csv")

### G.10 TreeSHAP nativo de XGBoost — `shap_pitching_plus.csv`

In [ ]:
import xgboost as xgb

model = res_full["model"]
booster = model.get_booster() if hasattr(model, "get_booster") else model
rng = np.random.default_rng(0)
idx = rng.choice(len(X_full), size=20000, replace=False)
Xs = X_full.iloc[idx]

# TreeSHAP nativo de XGBoost 
contrib = booster.predict(xgb.DMatrix(Xs.values), pred_contribs=True, validate_features=False)
sv = contrib[:, :-1]          # la última columna es el sesgo del modelo

imp = pd.DataFrame({"mean_abs_shap": np.abs(sv).mean(axis=0)}, index=Xs.columns)
imp["signo_sobre_xRV"] = [np.corrcoef(Xs[c], sv[:, i])[0, 1] if Xs[c].std() > 0 else np.nan
                          for i, c in enumerate(Xs.columns)]
imp["pct_total"] = 100 * imp["mean_abs_shap"] / imp["mean_abs_shap"].sum()
imp = imp.sort_values("mean_abs_shap", ascending=False)
print("TOP 15 VARIABLES (SHAP, Pitching+):"); print(imp.head(15).round(4))

stuff_cols = set(X_stuff.columns)
loc_cols = {"PlateLocHeight","PlateLocSide","in_strike_zone","Balls","Strikes"}
ars_cols = set(X_stuff_loc_ars.columns) - stuff_cols - loc_cols
def capa(c):
    return "Stuff" if c in stuff_cols else "Ubicación y conteo" if c in loc_cols else "Arsenal" if c in ars_cols else "Contexto, mano y altitud"
imp["capa"] = [capa(c) for c in imp.index]
por_capa = imp.groupby("capa")["pct_total"].sum().sort_values(ascending=False).round(1)
print("\nPESO POR CAPA (% del SHAP total):"); print(por_capa)
imp.round(5).to_csv("shap_pitching_plus.csv")

### G.11 R² por holdout — `r2_holdouts.csv`

In [ ]:
import numpy as np, pandas as pd
from sklearn.metrics import r2_score

y = df_cleaned["xRunValue"]
groups = df_cleaned["pitcher_anon_id"].fillna("unknown")
assert len(y) == len(pred_full) == len(X_full), (len(y), len(pred_full), len(X_full))

layers = {
    "Stuff":                       X_stuff,
    "Stuff + ubicación":           X_stuff_loc,
    "Stuff + ubicación + arsenal": X_stuff_loc_ars,
    "Pitching+ (completo)":        X_full,
}
yv = y.values
last = df_cleaned["year"].max()
yr, alt = df_cleaned["year"].values, df_cleaned["altitude_category"].values
masks = {
    "Temporada (última)": (yr != last, yr == last),
    "Altitud extrema":    (alt != "Extreme Altitude", alt == "Extreme Altitude"),
}

def fit_r2(X, tr, te):
    m = xgb.XGBRegressor(**XGB_KW)
    m.fit(X.iloc[np.where(tr)[0]], yv[tr])
    return r2_score(yv[te], m.predict(X.iloc[np.where(te)[0]]))

reuse = {"Stuff": pred_stuff, "Pitching+ (completo)": pred_full}
rows = []
for name, X in layers.items():
    oof = reuse.get(name)
    if oof is None:
        oof = oof_pred(X)
    row = {"Capa": name, "Lanzador (GroupKFold)": r2_score(yv, oof)}
    for k, (tr, te) in masks.items():
        row[k] = fit_r2(X, tr, te)
    rows.append(row); print(row)

R2 = pd.DataFrame(rows).set_index("Capa").round(4)
print("\nR² del xRunValue por capa y holdout:"); print(R2)
print(f"\nTemporada de prueba: {last} | pitcheos de prueba: temporada {int((yr==last).sum()):,}, altitud extrema {int((alt=='Extreme Altitude').sum()):,}")
R2.to_csv("r2_holdouts.csv")

### G.12 Auditoría final: holdout de juegos completos, bootstrap de la mejora y holdout de altitud

In [ ]:

# ============================================================
# CELDA FINAL — AUDITORÍA Y VALIDACIÓN
# con variables
# df_cleaned, X_stuff, X_stuff_loc, X_stuff_loc_ars,
# X_full_no_alt, X_full y BEST_PARAMS.
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

SEED = 2026
N_BOOT = 300

# ------------------------------------------------------------
# 1. Comprobaciones de integridad y alineacion
# ------------------------------------------------------------

required = [
    "game_anon_id", "pitcher_anon_id", "year",
    "altitude_category", "Outs", "Balls", "Strikes",
    "Outs_next", "Balls_next", "Strikes_next",
    "RunsScored", "runs_after"
]
missing = [c for c in required if c not in df_cleaned.columns]
if missing:
    raise KeyError(f"Faltan columnas: {missing}")

layers = {
    "Stuff+": X_stuff,
    "Stuff + Location": X_stuff_loc,
    "Stuff + Location + Arsenal": X_stuff_loc_ars,
    "Pitching+ sin altitud": X_full_no_alt,
    "Pitching+ completo": X_full,
}

d = df_cleaned.reset_index(drop=True).copy()
X_layers = {}

for name, X in layers.items():
    if len(X) != len(d) or not X.index.equals(df_cleaned.index):
        raise ValueError(
            f"Índices desalineados en {name}. "
            "No se ejecutará una evaluación potencialmente incorrecta."
        )
    X_layers[name] = X.reset_index(drop=True).replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)

print(f"Dataset: {len(d):,} pitcheos")
print(f"Juegos: {d['game_anon_id'].nunique():,}")
print(f"Lanzadores: {d['pitcher_anon_id'].nunique():,}")

# ------------------------------------------------------------
# 2. Funciones: reconstruir xRunValue con RE de entrenamiento
# ------------------------------------------------------------

STATE = ["Outs", "Balls", "Strikes"]
NEXT_STATE = ["Outs_next", "Balls_next", "Strikes_next"]

def fit_re_table(train):
    # RE se aprende únicamente con los juegos incluidos en train.
    return train.groupby(STATE)["runs_after"].mean()

def make_xrv(frame, re_table):
    out = frame.copy()

    current = pd.MultiIndex.from_frame(out[STATE])
    following = pd.MultiIndex.from_frame(out[NEXT_STATE])

    re_current = re_table.reindex(current).to_numpy()
    re_next = re_table.reindex(following).to_numpy()

    fallback = float(re_table.mean())
    re_current = np.where(np.isfinite(re_current), re_current, fallback)
    re_next = np.where(np.isfinite(re_next), re_next, fallback)

    return (
        re_next - re_current
        + out["RunsScored"].fillna(0).to_numpy(dtype=float)
    )

def fit_baseline(train, y_train, test):
    tmp = train[STATE].copy()
    tmp["_y"] = y_train
    table = tmp.groupby(STATE)["_y"].mean()

    pred = table.reindex(pd.MultiIndex.from_frame(test[STATE])).to_numpy()
    return np.where(np.isfinite(pred), pred, float(np.mean(y_train)))

def metrics(y, pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y, pred)),
        "MAE": mean_absolute_error(y, pred),
        "R2": r2_score(y, pred),
        "Spearman": spearmanr(y, pred).statistic,
    }

def run_comparison(train, test, feature_layers, tag):
    # Recalcular el target usando únicamente la RE de train.
    re_table = fit_re_table(train)
    ytr = make_xrv(train, re_table)
    yte = make_xrv(test, re_table)

    predictions = {
        "Baseline por conteo": fit_baseline(train, ytr, test)
    }
    fitted = {}

    for name, X in feature_layers.items():
        model = XGBRegressor(**BEST_PARAMS)
        model.fit(X.loc[train.index], ytr)
        predictions[name] = model.predict(X.loc[test.index])
        fitted[name] = model

    base_rmse = metrics(yte, predictions["Baseline por conteo"])["RMSE"]
    rows = []

    for name, pred in predictions.items():
        row = {"Validación": tag, "Modelo": name, **metrics(yte, pred)}
        row["Delta RMSE vs baseline"] = base_rmse - row["RMSE"]
        rows.append(row)

    return pd.DataFrame(rows), predictions, yte, fitted

# ------------------------------------------------------------
# 3. HOLDOUT DE JUEGOS COMPLETOS
# Ningun test en el entrenamiento.
# ------------------------------------------------------------

game_ids = d["game_anon_id"].astype(str).to_numpy()
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=SEED
)
itr, ite = next(splitter.split(d, groups=game_ids))

train = d.iloc[itr].copy()
test = d.iloc[ite].copy()

assert set(train["game_anon_id"].astype(str)).isdisjoint(
    set(test["game_anon_id"].astype(str))
)

# run_comparison usa indices originales para seleccionar X.
# Reiniciar indices aquí hace coincidir las posiciones de X y d.
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

# Construir una copia alineada de X para este split.
X_game = {}
for name, X in X_layers.items():
    X_game[name] = pd.concat(
        [X.iloc[itr].reset_index(drop=True),
         X.iloc[ite].reset_index(drop=True)],
        ignore_index=True
    )

combined = pd.concat([train, test], ignore_index=True)
ntr = len(train)

# Orden de filas: primero train, luego test.
game_results, game_preds, y_test_game, game_models = run_comparison(
    combined.iloc[:ntr].reset_index(drop=True),
    combined.iloc[ntr:].reset_index(drop=True),
    {name: X.reset_index(drop=True) for name, X in X_game.items()},
    "Holdout de juegos"
)

print("\n=== RESULTADOS: HOLDOUT DE JUEGOS ===")
print(f"Juegos train: {train['game_anon_id'].nunique():,}")
print(f"Juegos test: {test['game_anon_id'].nunique():,}")
print(game_results.sort_values("RMSE").round(5).to_string(index=False))

# ------------------------------------------------------------
# 4. INCERTIDUMBRE DE LA MEJORA VS BASELINE
# Bootstrap de juegos completos, no de pitcheos independientes.
# ------------------------------------------------------------

best_rows = game_results[
    game_results["Modelo"] != "Baseline por conteo"
].sort_values("RMSE")

best_name = best_rows.iloc[0]["Modelo"]
best_pred = game_preds[best_name]
base_pred = game_preds["Baseline por conteo"]

test_games = combined.iloc[ntr:]["game_anon_id"].astype(str).to_numpy()
unique_games = pd.unique(test_games)
indices_by_game = {
    g: np.flatnonzero(test_games == g) for g in unique_games
}

rng = np.random.default_rng(SEED)
deltas = []

for _ in range(N_BOOT):
    sampled = rng.choice(unique_games, size=len(unique_games), replace=True)
    ix = np.concatenate([indices_by_game[g] for g in sampled])

    rmse_base = np.sqrt(mean_squared_error(y_test_game[ix], base_pred[ix]))
    rmse_model = np.sqrt(mean_squared_error(y_test_game[ix], best_pred[ix]))
    deltas.append(rmse_base - rmse_model)

ci_low, ci_high = np.percentile(deltas, [2.5, 97.5])
delta = (
    np.sqrt(mean_squared_error(y_test_game, base_pred))
    - np.sqrt(mean_squared_error(y_test_game, best_pred))
)

print("\n=== INCERTIDUMBRE ===")
print("Mejor modelo en este split:", best_name)
print(f"Mejora RMSE vs baseline: {delta:+.5f}")
print(f"IC bootstrap 95% por juego: [{ci_low:+.5f}, {ci_high:+.5f}]")
if ci_low > 0:
    print("La mejora frente al baseline es consistente con este intervalo.")
elif ci_high < 0:
    print("El baseline supera al modelo en este intervalo.")
else:
    print("El intervalo incluye cero: la mejora no es concluyente.")

# ------------------------------------------------------------
# 5. HOLDOUT DE ALTITUD EXTREMA
# Prueba complementaria: excluir la altitud del modelo.
# No usar altitude_ord para pretender extrapolar un nivel no visto.
# ------------------------------------------------------------

extreme = d["altitude_category"].astype(str).eq("Extreme Altitude").to_numpy()
non_extreme = ~extreme

if extreme.sum() == 0 or non_extreme.sum() == 0:
    print("\nNo se pudo ejecutar el holdout de altitud: faltan categorías.")
else:
    train_alt = d.loc[non_extreme].copy().reset_index(drop=True)
    test_alt = d.loc[extreme].copy().reset_index(drop=True)

    X_alt = X_layers["Pitching+ sin altitud"]
    X_train_alt = X_alt.loc[non_extreme].reset_index(drop=True)
    X_test_alt = X_alt.loc[extreme].reset_index(drop=True)

    re_alt = fit_re_table(train_alt)
    y_train_alt = make_xrv(train_alt, re_alt)
    y_test_alt = make_xrv(test_alt, re_alt)

    baseline_alt = fit_baseline(train_alt, y_train_alt, test_alt)

    model_alt = XGBRegressor(**BEST_PARAMS)
    model_alt.fit(X_train_alt, y_train_alt)
    pred_alt = model_alt.predict(X_test_alt)

    result_alt = pd.DataFrame([
        {"Modelo": "Baseline por conteo", **metrics(y_test_alt, baseline_alt)},
        {"Modelo": "Pitching+ sin altitud", **metrics(y_test_alt, pred_alt)}
    ])
    base_rmse_alt = result_alt.loc[
        result_alt["Modelo"] == "Baseline por conteo", "RMSE"
    ].iloc[0]
    result_alt["Delta RMSE vs baseline"] = (
        base_rmse_alt - result_alt["RMSE"]
    )

    print("\n=== HOLDOUT DE ALTITUD EXTREMA ===")
    print(f"Train: {len(train_alt):,} | Test: {len(test_alt):,}")
    print("Altitudes train:",
          sorted(train_alt["altitude_category"].dropna().astype(str).unique()))
    print("Altitudes test:",
          sorted(test_alt["altitude_category"].dropna().astype(str).unique()))
    print(result_alt.round(5).to_string(index=False))

# ------------------------------------------------------------
# 6. AUDITORÍA DE VARIABLES Y LIMITACIONES
# ------------------------------------------------------------

print("\n=== AUDITORÍA FINAL ===")
print("Solapamiento de juegos en el holdout:",
      len(set(train["game_anon_id"]) & set(test["game_anon_id"])))
print("Columnas de estadio disponibles:",
      [c for c in d.columns if "stadium" in c.lower() or "park_id" in c.lower()])

print("""
INTERPRETACIÓN:
- Delta RMSE positivo: mejora respecto al baseline.
- Un IC que incluye cero no demuestra una mejora concluyente.
- El holdout de juegos no sustituye el holdout por lanzador.
- La categoría de altitud no equivale a un identificador de estadio.
- Esta celda recalcula xRunValue con RE de entrenamiento, pero las variables
  agregadas previamente (p. ej., pitch_usage_pct y diferencias contra la recta)
  también deben construirse dentro de cada partición para descartar toda fuga.
- No se debe afirmar que existe generalización a estadios no vistos sin un ID
  de estadio y una validación que retenga estadios completos.
""")

game_results.to_csv("validacion_final_holdout_juegos.csv", index=False)
print("Archivo guardado: validacion_final_holdout_juegos.csv")


### G.13 Validación corregida: holdout por juego con tabla RE y baseline solo de entrenamiento

In [ ]:

# ============================================================
# VALIDACIÓN CORREGIDA: HOLDOUT POR JUEGO
# - Alineación explícita de X e y mediante posiciones
# - xRunValue con tabla RE estimada solo en entrenamiento
# - Baseline por conteo entrenado solo en entrenamiento
# - Comparación con mejora RMSE = RMSE_baseline - RMSE_modelo
#
# Requiere:
# df_cleaned, df_encoded, X_stuff, X_stuff_loc,
# X_stuff_loc_ars, X_full_no_alt, X_full y BEST_PARAMS.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)
from scipy.stats import spearmanr
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Comprobar alineación y columnas necesarias
# ------------------------------------------------------------

required = [
    "game_anon_id", "Outs", "Balls", "Strikes",
    "RunsScored", "runs_after",
    "Outs_next", "Balls_next", "Strikes_next"
]

missing = [c for c in required if c not in df_cleaned.columns]
if missing:
    raise ValueError(f"Faltan columnas en df_cleaned: {missing}")

layers = {
    "Stuff+": X_stuff,
    "Stuff + Location": X_stuff_loc,
    "Stuff + Location + Arsenal": X_stuff_loc_ars,
    "Pitching+ sin altitud": X_full_no_alt,
    "Pitching+ completo": X_full
}

n = len(df_cleaned)

if len(df_encoded) != n:
    raise ValueError(
        f"Desalineación: df_cleaned tiene {n:,} filas y "
        f"df_encoded tiene {len(df_encoded):,}."
    )

for name, X in layers.items():
    if len(X) != n:
        raise ValueError(
            f"{name}: tiene {len(X):,} filas, pero df_cleaned tiene {n:,}."
        )

if df_cleaned["game_anon_id"].isna().any():
    raise ValueError("Hay juegos sin identificador; revisar antes de dividir.")

print(f"Filas verificadas: {n:,}")
print(f"Juegos: {df_cleaned['game_anon_id'].nunique():,}")
print("Las matrices tienen el mismo número de filas y se indexarán por posición.")

# ------------------------------------------------------------
# 2. Holdout por juego: ningún juego aparece en ambos grupos
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026
)

idx_train, idx_test = next(
    splitter.split(
        np.zeros(n),
        groups=df_cleaned["game_anon_id"].to_numpy()
    )
)

d = df_cleaned.reset_index(drop=True).copy()
train = d.iloc[idx_train].copy()
test = d.iloc[idx_test].copy()

games_train = set(train["game_anon_id"].unique())
games_test = set(test["game_anon_id"].unique())
overlap = games_train.intersection(games_test)

assert len(overlap) == 0, f"Hay juegos compartidos: {len(overlap)}"

print(f"\nJuegos train: {len(games_train):,}")
print(f"Juegos test:  {len(games_test):,}")
print(f"Pitcheos train: {len(train):,}")
print(f"Pitcheos test:  {len(test):,}")
print(f"Solapamiento de juegos: {len(overlap)}")

# ------------------------------------------------------------
# 3. Ajustar la tabla RE exclusivamente con entrenamiento
# ------------------------------------------------------------

state_cols = ["Outs", "Balls", "Strikes"]
next_state_cols = ["Outs_next", "Balls_next", "Strikes_next"]

for frame in (train, test):
    for c in state_cols + next_state_cols:
        frame[c] = pd.to_numeric(frame[c], errors="raise").astype(int)

# runs_after se calcula dentro de cada media entrada.
# Como separamos juegos completos, las secuencias de entrenamiento
# no contienen pitcheos de juegos de prueba.
re_train = (
    train.groupby(state_cols, observed=True)["runs_after"]
    .mean()
)

# Respaldo para estados ausentes en entrenamiento:
# media de los estados observados con el mismo num de outs.
re_by_outs = train.groupby("Outs", observed=True)["runs_after"].mean()
re_global = float(train["runs_after"].mean())

def lookup_re(frame, cols):
    keys = pd.MultiIndex.from_frame(frame[cols])
    vals = re_train.reindex(keys).to_numpy(dtype=float)

    missing_mask = np.isnan(vals)
    if missing_mask.any():
        outs_values = frame[cols[0]].to_numpy()
        fallback = np.array([
            re_by_outs.get(o, re_global) for o in outs_values
        ], dtype=float)
        vals[missing_mask] = fallback[missing_mask]

    # Estado terminal: tres outs, sin expectativa de carreras restante.
    terminal = frame[cols[0]].to_numpy() >= 3
    vals[terminal] = 0.0
    return vals

def make_xrunvalue(frame):
    re_current = lookup_re(frame, state_cols)
    re_next = lookup_re(frame, next_state_cols)
    runs_scored = pd.to_numeric(
        frame["RunsScored"], errors="coerce"
    ).fillna(0).to_numpy(dtype=float)

    return re_next - re_current + runs_scored

y_train = make_xrunvalue(train)
y_test = make_xrunvalue(test)

if not (np.isfinite(y_train).all() and np.isfinite(y_test).all()):
    raise ValueError("xRunValue contiene valores no finitos.")

print("\nxRunValue recalculado con RE de entrenamiento:")
print(f"Media train: {y_train.mean():+.5f}")
print(f"Media test:  {y_test.mean():+.5f}")

# ------------------------------------------------------------
# 4. Baseline por conteo: solo aprende del entrenamiento
# ------------------------------------------------------------

baseline_table = (
    pd.DataFrame({
        "Outs": train["Outs"].to_numpy(),
        "Balls": train["Balls"].to_numpy(),
        "Strikes": train["Strikes"].to_numpy(),
        "target": y_train
    })
    .groupby(state_cols, observed=True)["target"]
    .mean()
)

test_keys = pd.MultiIndex.from_frame(test[state_cols])
pred_baseline = baseline_table.reindex(test_keys).to_numpy(dtype=float)
pred_baseline = np.where(
    np.isnan(pred_baseline),
    float(np.mean(y_train)),
    pred_baseline
)

rmse_baseline = float(
    np.sqrt(mean_squared_error(y_test, pred_baseline))
)

def metrics(y_true, pred):
    rho = spearmanr(y_true, pred)[0]
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "MAE": mean_absolute_error(y_true, pred),
        "R2": r2_score(y_true, pred),
        "Spearman": rho
    }

rows = []
base_metrics = metrics(y_test, pred_baseline)
rows.append({
    "Validación": "Holdout de juegos",
    "Modelo": "Baseline por conteo",
    **base_metrics,
    "Mejora RMSE vs baseline": 0.0
})

# ------------------------------------------------------------
# 5. Entrenar y evaluar cada modelo con las mismas filas
# ------------------------------------------------------------

for name, X in layers.items():
    # Selección por posición: evita usar índices reiniciados o
    # etiquetas que no correspondan a los pitcheos de prueba.
    X_train = X.iloc[idx_train].copy()
    X_test = X.iloc[idx_test].copy()

    assert len(X_train) == len(y_train)
    assert len(X_test) == len(y_test)

    model = XGBRegressor(**BEST_PARAMS)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    result = metrics(y_test, pred)
    rows.append({
        "Validación": "Holdout de juegos",
        "Modelo": name,
        **result,
        "Mejora RMSE vs baseline": rmse_baseline - result["RMSE"]
    })

results = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)

print("\n=== RESULTADOS CORREGIDOS: HOLDOUT DE JUEGOS ===")
print("Mejora RMSE positiva = el modelo mejora al baseline.\n")
print(results.round(5).to_string(index=False))

# ------------------------------------------------------------
# 6. Auditoría final: agregados potencialmente contaminados
# ------------------------------------------------------------

aggregate_candidates = [
    "fb_velo", "fb_ivb", "fb_hb",
    "velo_diff_vs_fb", "ivb_diff_vs_fb", "hb_diff_vs_fb",
    "pitch_usage_pct", "tunnel_similarity"
]

present = [c for c in aggregate_candidates if c in d.columns]

print("\n=== AUDITORÍA DE VARIABLES ===")
print("Variables agregadas que deben revisarse:")
print(present if present else "No se encontraron esas columnas en df_cleaned.")

print("""
IMPORTANTE:
1. Esta celda corrige la selección de X por posición y ajusta la tabla RE
   con los juegos de entrenamiento.
2. No reconstruye las variables agregadas dentro de la partición.
   Si se calcularon usando todo el dataset, aún pueden introducir fuga.
3. Por tanto, estos resultados son una evaluación corregida de la alineación
   y de la tabla RE, pero no constituyen todavía una auditoría completa
   de fuga de información.
4. La categoría de altitud no identifica un estadio individual.
""")

results.to_csv(
    "validacion_holdout_juegos_corregida.csv",
    index=False,
    encoding="utf-8-sig"
)
print("Archivo guardado: validacion_holdout_juegos_corregida.csv")


### G.14 Baseline contra modelos — `baseline_vs_modelos.csv`

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

y = df_cleaned["xRunValue"]; yv = y.values
groups = df_cleaned["pitcher_anon_id"].fillna("unknown")
yr, alt = df_cleaned["year"].values, df_cleaned["altitude_category"].values
last = yr.max()
k_count     = (df_cleaned["Balls"]*10 + df_cleaned["Strikes"]).astype(int).values
k_count_out = k_count*10 + df_cleaned["Outs"].astype(int).values

splits = {
    "Lanzador (GroupKFold)": list(GroupKFold(5).split(X_full, yv, groups)),
    "Temporada (última)":    [(np.where(yr != last)[0], np.where(yr == last)[0])],
    "Altitud extrema":       [(np.where(alt != "Extreme Altitude")[0], np.where(alt == "Extreme Altitude")[0])],
}
def base_r2(keys, sp):
    yt, yp = [], []
    for tr, te in sp:
        mu = pd.Series(yv[tr]).groupby(keys[tr]).mean()
        yp.append(pd.Series(keys[te]).map(mu).fillna(yv[tr].mean()).values); yt.append(yv[te])
    return r2_score(np.concatenate(yt), np.concatenate(yp))
def model_r2(X, sp):
    yt, yp = [], []
    for tr, te in sp:
        m = xgb.XGBRegressor(**XGB_KW).fit(X.iloc[tr], yv[tr])
        yp.append(m.predict(X.iloc[te])); yt.append(yv[te])
    return r2_score(np.concatenate(yt), np.concatenate(yp))

X_cs = X_stuff.assign(Balls=df_cleaned["Balls"].values, Strikes=df_cleaned["Strikes"].values)
rows = {
    "BASELINE: xRV medio por conteo":         {k: base_r2(k_count, s) for k, s in splits.items()},
    "BASELINE: xRV medio por conteo y outs":  {k: base_r2(k_count_out, s) for k, s in splits.items()},
    "Conteo + Stuff (modelo)":                {k: model_r2(X_cs, s) for k, s in splits.items()},
}
T = pd.DataFrame(rows).T
for capa in ["Stuff", "Stuff + ubicación", "Pitching+ (completo)"]:
    if "R2" in globals() and capa in R2.index:
        T.loc[capa] = R2.loc[capa, list(splits)].values
print(T.round(4)); T.round(4).to_csv("baseline_vs_modelos.csv")

### G.15 Calibración por deciles — `calibracion_por_deciles.csv`

In [ ]:
import numpy as np, pandas as pd

yv = df_cleaned["xRunValue"].values
out = []
for nombre, pred in [("Stuff", pred_stuff), ("Pitching+ (completo)", pred_full)]:
    d = pd.DataFrame({"pred": pred, "real": yv})
    d["decil"] = pd.qcut(d["pred"], 10, labels=False, duplicates="drop") + 1   # 1 = mejores pitcheos esperados
    t = d.groupby("decil").agg(pred_media=("pred","mean"), real_media=("real","mean"), n=("real","size"))
    t["real_x100"] = t["real_media"] * 100        # carreras por 100 pitcheos
    t["modelo"] = nombre
    out.append(t.reset_index())
    print(f"\n{nombre}: calibración por decil (1 = mejor esperado)"); print(t[["pred_media","real_media","real_x100"]].round(4))
    spread = t.loc[t.index.max(), "real_x100"] - t.loc[t.index.min(), "real_x100"]
    print(f"Diferencia real decil 10 − decil 1: {spread:+.2f} carreras por 100 pitcheos")
    print(f"Correlación predicho vs. real entre deciles: {np.corrcoef(t.pred_media, t.real_media)[0,1]:.3f}")
pd.concat(out).to_csv("calibracion_por_deciles.csv", index=False)

### G.16 Location+ v2 (valor marginal de la ubicación) y exportes por pitcheo, lanzador y arsenal

In [ ]:
import os, numpy as np, pandas as pd

y = df_cleaned["xRunValue"]; yv = y.values
groups = df_cleaned["pitcher_anon_id"].fillna("unknown")

# --- Location+ v2: ubicación + conteo + tipo de pitch + mano del bateador (definición del brief) ---
base = ["PlateLocHeight", "PlateLocSide", "in_strike_zone", "Balls", "Strikes"]
print("X_loc_only actual:", X_loc_only.shape)
X_loc2 = X_full[base + dum].copy()                       # sin duplicados: ubicación + conteo + tipo de pitch + mano del bateador
X_loc2["loc_inside_batter"] = -df_cleaned["PlateLocSide"].values * bat_sign   # + = hacia adentro para el bateador
assert not X_loc2.columns.duplicated().any()
print("X_loc2:", X_loc2.shape)
pred_loc2 = oof_pred(X_loc2)
print(f"R² lanzadores no vistos — Location+ v1: {r2_score(yv, pred_loc):.4f} → v2: {r2_score(yv, pred_loc2):.4f}")
S["location_plus"] = plus_global(pred_loc2)

# Valor MARGINAL de la ubicación: modelo con ubicación − modelo sin ubicación (mismo conteo, tipo y mano del bateador)
base0 = ["Balls", "Strikes"] + dum
pred_m0 = oof_pred(X_full[base0])
marg = pred_loc2 - pred_m0
print(f"R² sin ubicación: {r2_score(yv, pred_m0):.4f} | con ubicación: {r2_score(yv, pred_loc2):.4f}")
print("Marginal (xRV): media {:.4f}, desv. {:.4f}, rango [{:.3f}, {:.3f}]".format(marg.mean(), marg.std(), marg.min(), marg.max()))
S["location_plus"] = plus_global(marg)
print(S["location_plus"].describe().round(2))

# --- Exportaciones del entregable 1: por pitcheo, por lanzador y por arsenal ---
cols = ["PitchUID","year","pitcher_anon_id","AutoPitchType","altitude_category",
        "stuff_plus","location_plus","pitching_plus","pitching_plus_adj"]
S[cols].round(2).to_csv("scores_por_pitcheo.csv.gz", index=False, compression="gzip")

tot = S.groupby("pitcher_anon_id").size()
A = (S.groupby(["pitcher_anon_id","AutoPitchType"], observed=True)
       .agg(n=("stuff_plus","size"), stuff_plus=("stuff_plus","mean"), location_plus=("location_plus","mean"),
            pitching_plus=("pitching_plus","mean"), pitching_plus_adj=("pitching_plus_adj","mean")).reset_index())
A["uso_pct"] = 100 * A["n"] / A["pitcher_anon_id"].map(tot)
A = A[A["n"] >= 50].round(2)
A.to_csv("scores_por_arsenal_lanzador_tipo.csv", index=False)

for f in ["scores_por_pitcheo.csv.gz", "scores_por_arsenal_lanzador_tipo.csv"]:
    print(f"{f}: {os.path.getsize(f)/1e6:.1f} MB")
print(S[["stuff_plus","location_plus","pitching_plus"]].describe().round(2))

### G.17 Riesgo por tipo de pitch (HR, rodados, contacto fuerte) e hipótesis H3 y H4

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score

ALT = {"No Altitude": "sea", "Extreme Altitude": "ext"}
dfa = df_cleaned[df_cleaned["altitude_category"].isin(ALT)].copy()
dfa["alt"] = dfa["altitude_category"].map(ALT)

# ======== A) Riesgo por tipo de pitch: altitud extrema vs. sin altitud (dado balón en juego) ========
print("Valores de hit_type:", df_cleaned["hit_type"].dropna().unique()[:10])
bip = dfa[dfa["is_ball_in_play"] == 1].copy()
bip["hr"] = (bip["home_run"] == 1).astype(float)
bip["gb"] = bip["hit_type"].astype(str).str.lower().str.contains("ground").astype(float)
ev = bip[bip["ExitSpeed"].notna()].copy()
ev["hard"] = (ev["ExitSpeed"] >= 95).astype(float)       # contacto fuerte = velocidad de salida >= 95 mph

def comp(flag, d, min_n=300):
    g = d.groupby(["AutoPitchType","alt"], observed=True)[flag].agg(["mean","count"]).unstack("alt")
    ps, pe, ns, ne = g[("mean","sea")], g[("mean","ext")], g[("count","sea")], g[("count","ext")]
    out = pd.DataFrame({"n_sea": ns, "n_ext": ne, "sea_%": 100*ps, "ext_%": 100*pe, "dif_pp": 100*(pe-ps),
                        "±IC95_pp": 196*np.sqrt(ps*(1-ps)/ns + pe*(1-pe)/ne)})
    return out[(ns >= min_n) & (ne >= min_n)].round(2)

print("\nHOME RUN por balón en juego (%):");        print(comp("hr", bip))
print("\nBOLA RODADA por balón en juego (%):");      print(comp("gb", bip))
print("\nCONTACTO FUERTE (>=95 mph) por balón en juego (%):"); print(comp("hard", ev))

# ======== B) H3: ¿los lanzadores de rodadores pierden menos en altitud? ========
assert "delta_pp" in R.columns, "R debe ser la tabla de Δ por lanzador (celda de cambio por altitud)"
gbp = bip.groupby("pitcher_anon_id").agg(n_bip=("gb","size"), gb_rate=("gb","mean")).query("n_bip >= 100")
H3 = R.join(gbp, how="inner")
H3["tercil_gb"] = pd.qcut(H3["gb_rate"], 3, labels=["menos rodadores","medio","más rodadores"])
print("\nH3 — Δ de Pitching+ en altitud extrema por tercil de tasa de rodadores:")
print(H3.groupby("tercil_gb", observed=True).agg(lanzadores=("delta_pp","size"), gb_rate=("gb_rate","mean"),
        delta_pp=("delta_pp","mean"), error_est=("delta_pp", lambda s: s.std()/np.sqrt(len(s)))).round(3))
print(f"Correlación tasa de rodadores vs. Δ Pitching+: {H3[['gb_rate','delta_pp']].corr().iloc[0,1]:.3f} (n = {len(H3)})")

# ======== C) H4: spin vs. forma del movimiento ========
cols = list(X_stuff.columns)
spin_c = [c for c in cols if c in ("SpinRate","SpinAxis")]
move_c = [c for c in cols if c in ("VertBreak","InducedVertBreak","HorzBreak","pfxx","pfxz","ax0","ay0","az0",
                                    "VertApprAngle","HorzApprAngle") or c.startswith("PitchTrajectory")]
print("\nspin:", spin_c, "| variables de movimiento:", len(move_c))
codes = pd.factorize(df_cleaned["pitcher_anon_id"].fillna("unknown"))[0]
idx_h = np.where(codes % 2 == 0)[0]                      # mitad de los lanzadores, para que sea rápido
yv_h = df_cleaned["xRunValue"].values
def r2_cv(X):
    Xs, ys, gs = X.iloc[idx_h], yv_h[idx_h], codes[idx_h]
    oof = np.zeros(len(idx_h))
    for tr, te in GroupKFold(3).split(Xs, ys, gs):
        m = xgb.XGBRegressor(**XGB_KW).fit(Xs.iloc[tr], ys[tr]); oof[te] = m.predict(Xs.iloc[te])
    return r2_score(ys, oof)
H4 = pd.Series({"Stuff completo": r2_cv(X_stuff),
                "Sin spin (SpinRate, SpinAxis)": r2_cv(X_stuff.drop(columns=spin_c)),
                "Solo spin, sin variables de movimiento": r2_cv(X_stuff.drop(columns=move_c))}).round(4)
print("\nH4 — R² del xRunValue en lanzadores no vistos (mitad de los lanzadores):"); print(H4)

# eficiencia de movimiento aproximada: pulgadas de movimiento total por cada 1,000 rpm
E = df_cleaned.loc[dfa.index, ["AutoPitchType","SpinRate","InducedVertBreak","HorzBreak"]].copy()
E["alt"] = dfa["alt"]
E["mov"] = np.sqrt(E["InducedVertBreak"]**2 + E["HorzBreak"]**2)
E["mov_1000rpm"] = E["mov"] / (E["SpinRate"] / 1000)
keep = E.groupby("AutoPitchType", observed=True).size().loc[lambda s: s > 5000].index
T = E[E["AutoPitchType"].isin(keep)].groupby(["AutoPitchType","alt"], observed=True)[["SpinRate","mov","mov_1000rpm"]].mean().unstack("alt")
print("\nEficiencia aproximada del movimiento (pulgadas por 1,000 rpm), sin altitud → extrema:")
print(pd.DataFrame({"spin_Δ_rpm": T[("SpinRate","ext")] - T[("SpinRate","sea")],
                    "mov_Δ_in": T[("mov","ext")] - T[("mov","sea")],
                    "in_x1000rpm_sea": T[("mov_1000rpm","sea")], "in_x1000rpm_ext": T[("mov_1000rpm","ext")]}).round(2))

### G.18 Lanzadores que más y menos pierden Stuff+ en altitud extrema — `top3_bottom3_*.csv`

In [28]:
import numpy as np, pandas as pd

COL = "Stuff_plus_pure"
MIN_T = 30    # pitcheos mínimos por tipo de pitch en CADA entorno
MIN_P = 100   # pitcheos mínimos por lanzador en CADA entorno (suma de sus tipos)
feats = ["RelSpeed", "SpinRate", "InducedVertBreak", "HorzBreak", "Extension", "RelHeight", "VertApprAngle"]

d = df_eval[df_eval["altitude_category"].isin(["No Altitude", "Extreme Altitude"])]

# 1) Δ por lanzador y tipo (mismo lanzador, mismo tipo): extrema − sin altitud
g = (d.groupby(["pitcher_anon_id", "AutoPitchType", "altitude_category"])[COL]
       .agg(["mean", "var", "size"]).unstack("altitude_category"))
g.columns = [f"{a}_{'no' if b == 'No Altitude' else 'ex'}" for a, b in g.columns]
g = g.dropna()
g = g[(g["size_no"] >= MIN_T) & (g["size_ex"] >= MIN_T)].copy()
g["delta"] = g["mean_ex"] - g["mean_no"]
g["w"] = 1 / (g["var_ex"] / g["size_ex"] + g["var_no"] / g["size_no"])   # peso = 1/varianza
g["wd"] = g["w"] * g["delta"]

# 2) Δ por lanzador = promedio ponderado de sus tipos
S = g.groupby(level=0).agg(wd=("wd", "sum"), w=("w", "sum"), n_no=("size_no", "sum"), n_ex=("size_ex", "sum"))
S["delta"] = S["wd"] / S["w"]
S["se"] = 1 / np.sqrt(S["w"])
S["tipos"] = g.reset_index().groupby("pitcher_anon_id")["AutoPitchType"].agg(lambda s: ", ".join(sorted(s)))
S = S[(S["n_no"] >= MIN_P) & (S["n_ex"] >= MIN_P)].copy()

# 3) encogimiento empírico-Bayes hacia el promedio
mu = np.average(S["delta"], weights=1 / S["se"] ** 2)
tau2 = max(0.0, S["delta"].var() - (S["se"] ** 2).mean())
S["delta_shrunk"] = mu + (tau2 / (tau2 + S["se"] ** 2)) * (S["delta"] - mu)
S["ic95"] = 1.96 * S["se"]
print(f"Lanzadores elegibles: {len(S)} | Δ Stuff+ medio {mu:+.2f} | tau = {np.sqrt(tau2):.2f}")

cols = ["delta", "ic95", "delta_shrunk", "n_no", "n_ex", "tipos"]
top = S.nlargest(3, "delta_shrunk");  top["grupo"] = "menos pierden"
bot = S.nsmallest(3, "delta_shrunk"); bot["grupo"] = "más pierden"
print("\nMENOS PIERDEN Stuff+ en altitud extrema:"); print(top[cols].round(2).to_string())
print("\nMÁS PIERDEN Stuff+ en altitud extrema:");  print(bot[cols].round(2).to_string())

# 4) perfil físico (sin altitud) de su tipo principal, como diferencia contra el promedio de la liga en ese tipo
rows = []
for pid, r in pd.concat([top, bot]).iterrows():
    sub = d[(d["pitcher_anon_id"] == pid) & (d["altitude_category"] == "No Altitude")]
    pt = sub["AutoPitchType"].value_counts().index[0]
    mine = sub[sub["AutoPitchType"] == pt]
    lg = d[(d["AutoPitchType"] == pt) & (d["altitude_category"] == "No Altitude")]
    row = {"pitcher": pid, "grupo": r["grupo"], "tipo_principal": pt, "n": len(mine)}
    for f in feats:
        row[f] = mine[f].mean() - lg[f].mean()
    rows.append(row)
perfil = pd.DataFrame(rows)
print("\nPerfil físico vs promedio de la liga en su tipo principal (sin altitud):")
print(perfil.round(2).to_string(index=False))

pd.concat([top, bot])[["grupo"] + cols].reset_index().round(3).to_csv("top3_bottom3_stuff_altitud.csv", index=False)
perfil.round(3).to_csv("top3_bottom3_perfil_fisico.csv", index=False)

C:\Users\jorge\AppData\Local\Temp\ipykernel_18084\1307643504.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = (d.groupby(["pitcher_anon_id", "AutoPitchType", "altitude_category"])[COL]


Lanzadores elegibles: 312 | Δ Stuff+ medio -6.28 | tau = 1.95

MENOS PIERDEN Stuff+ en altitud extrema:
                 delta  ic95  delta_shrunk  n_no  n_ex                      tipos
pitcher_anon_id                                                                  
pitcher_00477     0.26  1.29         -0.41   665   151          Four-Seam, Sinker
pitcher_00547    -0.98  1.78         -1.93   206   202  Four-Seam, Sinker, Slider
pitcher_00053    -1.50  1.58         -2.20   167   283          Four-Seam, Slider

MÁS PIERDEN Stuff+ en altitud extrema:
                 delta  ic95  delta_shrunk  n_no  n_ex                     tipos
pitcher_anon_id                                                                 
pitcher_00699   -12.97  1.55        -12.03   168   347      Curveball, Four-Seam
pitcher_00373   -11.66  0.92        -11.36   613   444  Changeup, Sinker, Slider
pitcher_01060   -11.29  1.03        -10.95   209   955      Curveball, Four-Seam

Perfil físico vs promedio de la liga en 